# NewsSeek: 166K Keyframes CLIP vs SigLIP Feature Encoding & Multilingual Benchmark

This notebook performs:
1. GPU batch feature extraction for 166,241 scene-aware keyframes using `openai/clip-vit-base-patch32` and `google/siglip-base-patch16-224`.
2. Multilingual query encoding for 698 video events (EN, VI, JA, etc.).
3. Exact FAISS `IndexFlatIP` vector store construction and artifact packaging for live search.
4. Video-wise Max-Pooling Cross-Modal Retrieval Benchmark (Recall@1, 5, 10, MRR, Modality Gap).

In [ ]:
!pip install -q faiss-cpu transformers

import os
import sys
import time
import json
import zipfile
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import faiss
from transformers import AutoProcessor, AutoModel, CLIPProcessor, CLIPModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB')


In [ ]:
# 698 Video Metadata Records (ID, Title, Language, Source)
VIDEO_METADATA = json.loads('{"gqZ2ArUOv3E": {"title": "Nightline Full Broadcast — Monday, August 10, 2026", "language": "en", "source_name": "ABC News"}, "G14g1DS2GSI": {"title": "Luigi Mangione arrives in state court ahead of September trial", "language": "en", "source_name": "ABC News"}, "ASrkeo2-fz4": {"title": "Mangione in court as trial nears", "language": "en", "source_name": "ABC News"}, "1Tv2tS60avM": {"title": "Trump signs executive order changing childhood vaccine schedule", "language": "en", "source_name": "ABC News"}, "L5Ld4XpPj5w": {"title": "Russia can\'t wage war without North Korea, Zelenskyy says", "language": "en", "source_name": "ABC News"}, "Na281-X4ykc": {"title": "Good Morning America: News to start your day | August 11", "language": "en", "source_name": "ABC News"}, "J-S49UW7u3o": {"title": "Death toll rises in Colombia earthquake", "language": "en", "source_name": "ABC News"}, "DptCEEN64Eg": {"title": "Doctors push back on Trump executive order to change childhood vaccine schedule", "language": "en", "source_name": "ABC News"}, "zWuq3q7Lycg": {"title": "Brad Pitt opens up about sobriety, suicidal thoughts in candid interview", "language": "en", "source_name": "ABC News"}, "0nz0tbiEjl0": {"title": "Trump Snuck Off Air Force One. What You Need To Know - August 11", "language": "en", "source_name": "ABC News"}, "nxcbn3WcLA8": {"title": "Who shot Tupac? Jury selection underway in the trial of Duane \'Keffe D\' Davis", "language": "en", "source_name": "ABC News"}, "FTU0DV74soU": {"title": "Trump reportedly used catering truck to sneak off Air Force One amid Iranian threats", "language": "en", "source_name": "ABC News"}, "eAvHslExQJE": {"title": "The legacy of Tupac Shakur 30 years after his murder", "language": "en", "source_name": "ABC News"}, "PZl8UAbO6pE": {"title": "Iran expert on Iran war stalemate: Hurting US more than Iran", "language": "en", "source_name": "ABC News"}, "uAmQiwaZcB0": {"title": "Major retailers pull products made with potentially contaminated jalapeños", "language": "en", "source_name": "ABC News"}, "0ycUwkTrT5Q": {"title": "Urgent rescue operation after deadly magnitude 7.4 earthquake strikes Colombia", "language": "en", "source_name": "ABC News"}, "hqocY88QOZk": {"title": "Urgent rescue operation after deadly 7.4 magnitude earthquake strikes Colombia", "language": "en", "source_name": "ABC News"}, "etXjH2O1z9g": {"title": "American Airlines pilot declares mayday after bird strike", "language": "en", "source_name": "ABC News"}, "YNL5kVgw7tg": {"title": "HHS senior advisor defends Trump administration\'s health policies", "language": "en", "source_name": "ABC News"}, "OJtS81FAYyA": {"title": "President Trump acknowledges \'no video or proof\' of alleged vandalism to Reflecting Pool", "language": "en", "source_name": "ABC News"}, "APbhHarRnT0": {"title": "Councils to get more powers to stop vape and betting shops | BBC News", "language": "en", "source_name": "BBC News"}, "-OMplfI3OZA": {"title": "Russia using North Korean missiles to strike Ukraine, Zelensky says | BBC News", "language": "en", "source_name": "BBC News"}, "jxj8eHm1VGA": {"title": "China’s lonely men are being conned by fake marriages | BBC News", "language": "en", "source_name": "BBC News"}, "fvitQ9eATuU": {"title": "At least 132 killed in Colombia\'s largest earthquake in years | BBC News", "language": "en", "source_name": "BBC News"}, "hLh9tQAzGUc": {"title": "President Trump demands compensation from Iran | BBC News", "language": "en", "source_name": "BBC News"}, "6KAXp0UzU8o": {"title": "Solar eclipse: When is it and how to watch it safely | BBC News", "language": "en", "source_name": "BBC News"}, "ZfEka-1wUOs": {"title": "Will \'Islamic Nato\' protect Pakistan, Saudi Arabia and Turkey? | Global News Podcast", "language": "en", "source_name": "BBC News"}, "PJxy5qj4HhE": {"title": "At least 13 killed in Ukrainian drone strike deep into Russia | BBC News", "language": "en", "source_name": "BBC News"}, "lcmnMMXJtDs": {"title": "World\'s oceans hit record-high July temperatures, EU\'s climate monitoring service says | BBC News", "language": "en", "source_name": "BBC News"}, "JyikzZCyi2Q": {"title": "Travelodge gave room key to domestic abuse victim\'s attacker | BBC News", "language": "en", "source_name": "BBC News"}, "RyXsMtbfVKQ": {"title": "Netanyahu rejects Trump’s Gaza peace plan | BBC News", "language": "en", "source_name": "BBC News"}, "71KzB-HHaGE": {"title": "Israel rejects Trump\'s 15-point plan for Gaza, Netanyahu says | BBC News", "language": "en", "source_name": "BBC News"}, "ri8FbguG7S0": {"title": "Inside the Chinese factory using robots to power online shopping | BBC News", "language": "en", "source_name": "BBC News"}, "W3wWiOvMB98": {"title": "Emergency declared as Canada wildfire doubles in size | BBC News", "language": "en", "source_name": "BBC News"}, "iU-0T_WmI-Q": {"title": "Fifa boss Gianni Infantino denies claims Uefa paid off alleged \'lover\' | BBC News", "language": "en", "source_name": "BBC News"}, "aWi51k5syfk": {"title": "Three killed in Russian missile attacks near Kyiv | BBC News", "language": "en", "source_name": "BBC News"}, "bCjAqK1IRt4": {"title": "How this rehab course is changing teen hackers for good | BBC News", "language": "en", "source_name": "BBC News"}, "-tQILE3ZUvw": {"title": "Saudi Arabia, Turkey and Pakistan make pact as tensions grow in Strait of Hormuz | BBC News", "language": "en", "source_name": "BBC News"}, "eMeSw1h_R6o": {"title": "Trump fury as US court halts his $400m dollar ballroom project | BBC News", "language": "en", "source_name": "BBC News"}, "z9FXO6_0Nv0": {"title": "AI just created a brand new virus. Should we be scared? | BBC News", "language": "en", "source_name": "BBC News"}, "kMofov6_i40": {"title": "Tucker Carlson expresses his \'deep regret\' to Hunter Biden", "language": "en", "source_name": "CNN"}, "O4sBG32XN3Q": {"title": "Erin Burnett debunks Trump’s childhood vaccine claim", "language": "en", "source_name": "CNN"}, "ATyg3tTlxz0": {"title": "Trump targets birthright citizenship again: Legal experts react", "language": "en", "source_name": "CNN"}, "lvrtDau6mFA": {"title": "Trump used catering truck to take secret flight from Turkey: WaPo", "language": "en", "source_name": "CNN"}, "gfeVcR-1_GE": {"title": "Ukraine launches deadliest attack on Russia in 2 years: What we know", "language": "en", "source_name": "CNN"}, "CNL678kkg9A": {"title": "\'Expensive distraction\': Doctors balk at Trump\'s child vaccine overhaul", "language": "en", "source_name": "CNN"}, "IhZtYIRsUkY": {"title": "Are Gulf countries abandoning the US? Fareed discusses", "language": "en", "source_name": "CNN"}, "lnHL2sq1he8": {"title": "Childcare is unaffordable in America. So how do we fix it?", "language": "en", "source_name": "CNN"}, "Oha8foAcHzc": {"title": "El-Sayed on his priorities, friction with Dems & call with Obama", "language": "en", "source_name": "CNN"}, "fkvTeHypZWg": {"title": "Iran war: Oil prices climb as Trump is only \'semi-negotiating\'", "language": "en", "source_name": "CNN"}, "mK-cs-dvea4": {"title": "Netanyahu rejects Trump\'s \'historic\' Gaza plan: Analysts react", "language": "en", "source_name": "CNN"}, "pS5kSWLBkfk": {"title": "What evidence justified Operation Southern Spear? A CNN investigation", "language": "en", "source_name": "CNN"}, "Kr8GozmwDdw": {"title": "Midterm madness, Trump\'s blame game, the Dem revolution & more | Enten roundup", "language": "en", "source_name": "CNN"}, "kmYd5mjUMxE": {"title": "Will regular Americans get any of $100B in Trump tariff refunds?", "language": "en", "source_name": "CNN"}, "5Qpf9mJxYK4": {"title": "Hormuz, Oman deal, new defense pact & more | Military experts on Iran", "language": "en", "source_name": "CNN"}, "FZatljJ2yY8": {"title": "Trump\'s 5 worst second-term polling collapses", "language": "en", "source_name": "CNN"}, "o8cGNWYCNmE": {"title": "Trump’s top general wants a way out of the Iran war: Sources", "language": "en", "source_name": "CNN"}, "7HuimiAnBa0": {"title": "US intel warns Putin is planning a NATO attack: What we know", "language": "en", "source_name": "CNN"}, "JCCJGFcFex4": {"title": "Hidden text may help job applicants trick A.I. screeners", "language": "en", "source_name": "NBC News"}, "E1rxmIkQpec": {"title": "NBC Nightly News with Tom Llamas Full Episode - Aug. 10", "language": "en", "source_name": "NBC News"}, "jLFyDOPbrlE": {"title": "OnlyFans model accepts plea deal in boyfriend’s killing", "language": "en", "source_name": "NBC News"}, "LAGZ9dASGeo": {"title": "New food items recalled over salmonella outbreak", "language": "en", "source_name": "NBC News"}, "_IzGnfhVZPg": {"title": "Beloved bald eagle Jackie dies after weeks of intensive care", "language": "en", "source_name": "NBC News"}, "MtaTKzw93gQ": {"title": "Family’s sweet birthday tradition goes viral", "language": "en", "source_name": "NBC News"}, "OEMMRL7_Lww": {"title": "Colombia struck with 7.4 magnitude earthquake, killing over 100", "language": "en", "source_name": "NBC News"}, "p-GBdm7lOGA": {"title": "Tornadoes tear through Midwest as wildfires burn in the west", "language": "en", "source_name": "NBC News"}, "drBTX3UWY3k": {"title": "Trump signs order to change child vaccine recommendations", "language": "en", "source_name": "NBC News"}, "NtQByVwqRbc": {"title": "Brad Pitt opens up about thoughts of suicide amid ‘family stuff’", "language": "en", "source_name": "NBC News"}, "UkrK69ASTns": {"title": "Boat captain charged after vessel capsizes near Statue of Liberty, killing woman and her child", "language": "en", "source_name": "NBC News"}, "1zjizpDbLLA": {"title": "Psychiatrist \'surprised\' that Lindsay Clancy called suicide hotline months before alleged murders", "language": "en", "source_name": "NBC News"}, "h_P1AeeJs2E": {"title": "Man arrested after alleged burglary at Kim Kardashian\'s Hidden Hills property", "language": "en", "source_name": "NBC News"}, "oCarUP6UEes": {"title": "Connecticut Democrat challenging incumbent says leaders need to recognize ‘when to pass the torch’", "language": "en", "source_name": "NBC News"}, "i7wiall8GP0": {"title": "Connecticut Democrat facing primary challenger running on a ‘record of accomplishment & achievement’", "language": "en", "source_name": "NBC News"}, "-vARuqp602Y": {"title": "New D.C.exhibit celebrating features the art of Brendan Murphy", "language": "en", "source_name": "NBC News"}, "NgrIh5rAKLw": {"title": "Trump signs executive order on childhood vaccines", "language": "en", "source_name": "NBC News"}, "-60amYiOTOk": {"title": "Democratic Minnesota Senate candidate says her campaign is about ‘the many versus the money’", "language": "en", "source_name": "NBC News"}, "Zs9tvcCRYek": {"title": "Death toll in earthquake surpasses 100, Colombian president says", "language": "en", "source_name": "NBC News"}, "sSXnMyNI5Zg": {"title": "Major earthquake devastates western Colombia with at least 164 people dead", "language": "en", "source_name": "Sky News"}, "bt5gAz0jMnI": {"title": "\'The most amazing thing I have ever seen\': Watch the UK\'s last total solar eclipse", "language": "en", "source_name": "Sky News"}, "vduOpzLbzNA": {"title": "PM addresses high street reforms and early prison releases", "language": "en", "source_name": "Sky News"}, "Z9K1gatwI4Y": {"title": "US President comes to aid of embattled FIFA chief via Truth Social post", "language": "en", "source_name": "Sky News"}, "wv5r0Kt8eA0": {"title": "West Bank families facing settler attacks", "language": "en", "source_name": "Sky News"}, "Oe9_Trs049E": {"title": "Boat carrying 230 migrants arrives in the UK", "language": "en", "source_name": "Sky News"}, "fikaQPJ-Du0": {"title": "Aftershocks rock Colombia after major 7.4 magnitude earthquake", "language": "en", "source_name": "Sky News"}, "SSyWFkWPTCo": {"title": "Why Ukraine\'s missile race against Russia could decide who wins this war", "language": "en", "source_name": "Sky News"}, "uLJpadX_YEY": {"title": "Will you get to see Wednesday\'s solar eclipse?", "language": "en", "source_name": "Sky News"}, "yZS0y8GGeBY": {"title": "Don\'t risk permanent eye damage - here\'s how to safely watch Wednesday\'s eclipse", "language": "en", "source_name": "Sky News"}, "hZMh-Uz7jIg": {"title": "Powerful earthquake kills at least 20 people in Colombia", "language": "en", "source_name": "Sky News"}, "qJJ-odMl4ek": {"title": "Huge fire breaks out at industrial estate", "language": "en", "source_name": "Sky News"}, "JDe7dq6AqUY": {"title": "Bezos in final stages of securing deal to buy stake in Liverpool Football Club", "language": "en", "source_name": "Sky News"}, "YUUHRsj4-JM": {"title": "Nigel Farage issues warning after a record 230 migrants arrive on a single boat", "language": "en", "source_name": "Sky News"}, "pT2Vg500tXE": {"title": "Ukraine making Russia \'pay highest possible price\' - as Kyiv launches \'massive\' drone attack", "language": "en", "source_name": "Sky News"}, "loxglw8SYUc": {"title": "Three FIFA confederations issue joint statement attacking Gianni Infantino over World Cup plan", "language": "en", "source_name": "Sky News"}, "v0NFqYwp5b4": {"title": "Daniel Kinahan remanded in custody after return to Ireland", "language": "en", "source_name": "Sky News"}, "dK0QJml2dbk": {"title": "Alleged organised crime boss Daniel Kinahan arrives in Ireland", "language": "en", "source_name": "Sky News"}, "QWv58D-g5UA": {"title": "34 Days Above 30C: UK heatwave sparks food shortage fears", "language": "en", "source_name": "Sky News"}, "TY11z2YsAxY": {"title": "Netanyahu rejects Donald Trump\'s Board of Peace plan for Gaza", "language": "en", "source_name": "Sky News"}, "bYepIOt1EBw": {"title": "Emergency crews in Colombia desperately search for earthquake survivors", "language": "en", "source_name": "CBS News"}, "uMNxe8b7cwk": {"title": "When did Americans start doubting public health experts?", "language": "en", "source_name": "CBS News"}, "kA35Xqh1Hf0": {"title": "Meta\'s Mark Zuckerberg thinks AI model code should be open to all", "language": "en", "source_name": "CBS News"}, "PQS1jAJ1yqk": {"title": "Powerball jackpot grows to staggering $975 million with UK players adding to sum", "language": "en", "source_name": "CBS News"}, "_kz1YDS-uBE": {"title": "Darline Graham gets Trump support in South Carolina\'s primary to replace Lindsey Graham", "language": "en", "source_name": "CBS News"}, "qCp0ADmhmrs": {"title": "Fact-checking medical claims after Trump\'s new order on childhood vaccine recommendations", "language": "en", "source_name": "CBS News"}, "tzA5M_Aa9jg": {"title": "What we know about the threat against Trump\'s Air Force One", "language": "en", "source_name": "CBS News"}, "_zT9vmlK1v0": {"title": "Despair, hope in Colombia as rescue teams comb debris for earthquake survivors", "language": "en", "source_name": "CBS News"}, "ZgJnAPFH-2E": {"title": "Election races to keep an eye on with primaries underway in six states", "language": "en", "source_name": "CBS News"}, "8Ns8DtAr7YU": {"title": "Wildfires incinerate homes and ecosystems in the western U.S. and Canada", "language": "en", "source_name": "CBS News"}, "_uNRTxZUtTs": {"title": "Colombia earthquake rescue crews race to free trapped survivors", "language": "en", "source_name": "CBS News"}, "FKUl7sqXO8A": {"title": "How Trump secretly switched planes in Turkey amid Iran\'s death threats", "language": "en", "source_name": "CBS News"}, "LmqsWeJivbs": {"title": "Doctors respond to Trump\'s vaccine executive order as kids head back to school", "language": "en", "source_name": "CBS News"}, "tFqDAZ5YuSA": {"title": "Mark Zuckerberg\'s vision of superintelligent AI for everyone", "language": "en", "source_name": "CBS News"}, "vmWVWHZIOik": {"title": "How have the goals of the Iran war evolved?", "language": "en", "source_name": "CBS News"}, "4kUmPaSaSn4": {"title": "Taylor Farms recalls jalapeño products at retailers like Target and Walmart over salmonella risk", "language": "en", "source_name": "CBS News"}, "aqJhcwIvLH4": {"title": "Search for survivors underway, more than 100 dead after Colombia earthquake", "language": "en", "source_name": "CBS News"}, "AAVqoeJZUWw": {"title": "How Trump achieved his 2024 political comeback", "language": "en", "source_name": "CBS News"}, "qRIVIjto_qM": {"title": "Israel rejects Trump’s Gaza peace plan - what comes next?", "language": "en", "source_name": "Channel 4 News"}, "o8L5ZXBBdjs": {"title": "Over 70% of England in drought as UK braces for fifth heatwave", "language": "en", "source_name": "Channel 4 News"}, "RhfcZzt9fD8": {"title": "Government declares national emergency after major earthquake rips through Colombia", "language": "en", "source_name": "Channel 4 News"}, "Wp7sZ6HhODU": {"title": "Why Gen Z are getting anti-anxiety drugs from the dark web", "language": "en", "source_name": "Channel 4 News"}, "G19OQ03_lBc": {"title": "Alleged mob boss Daniel Kinahan returns to Ireland to face charges", "language": "en", "source_name": "Channel 4 News"}, "dc1hLu9zkUo": {"title": "UK heatwave to return next week as wildfire risk climbs", "language": "en", "source_name": "Channel 4 News"}, "tBC27Rx9lFQ": {"title": "Prolific sex offender guilty of double murder and rape", "language": "en", "source_name": "Channel 4 News"}, "MhWxu_RI_uw": {"title": "How AI could be taking on a life of its own", "language": "en", "source_name": "Channel 4 News"}, "X-qzrT7m9Ag": {"title": "Meta ordered to pay almost $1bn over social media harm to children", "language": "en", "source_name": "Channel 4 News"}, "xmYpSNJViM0": {"title": "Infantino clings to job after apology for World Cup sale proposal", "language": "en", "source_name": "Channel 4 News"}, "n3vcjcucpLQ": {"title": "Iran and Oman close to deal for Strait of Hormuz", "language": "en", "source_name": "Channel 4 News"}, "WgmVRo-1PA0": {"title": "Why did Cambridge professor Jason Arday resign?", "language": "en", "source_name": "Channel 4 News"}, "5iJejM8Evc0": {"title": "Who is the manosphere influencer linked to a fake weight loss drugs bust?", "language": "en", "source_name": "Channel 4 News"}, "wxPZumxbyCw": {"title": "El-Sayed wins in Michigan - will he unite the Democrats against Trump?", "language": "en", "source_name": "Channel 4 News"}, "ukIEfb5mG0Y": {"title": "Early release plan will put people at risk, says UK probation union", "language": "en", "source_name": "Channel 4 News"}, "OpXlGrKUPmo": {"title": "Russian missiles pound Kyiv as Ukraine targets ballistic sites", "language": "en", "source_name": "Channel 4 News"}, "rcPRsohSbDg": {"title": "Why Michigan’s senate race could decide the future of the Democrats", "language": "en", "source_name": "Channel 4 News"}, "6l-Vvvoa4wc": {"title": "Mass funeral in Gaza for 112 people killed in Israeli airstrikes", "language": "en", "source_name": "Channel 4 News"}, "hFeJu2CNioE": {"title": "Some prisoner releases to go ahead despite Burnham’s pause", "language": "en", "source_name": "Channel 4 News"}, "WNStuD_d07U": {"title": "Hasan Piker on UK entry ban, rise of far-right and claims of antisemitism - full interview", "language": "en", "source_name": "Channel 4 News"}, "KxxnMhtsWnk": {"title": "\'WOKE 1 WAS CRAZY\': AOC distances herself from Francesca Hong", "language": "en", "source_name": "Fox News"}, "oUSl0BA7nRw": {"title": "He was \'LYING\': OUTRAGE after bombshell Fauci texts revealed", "language": "en", "source_name": "Fox News"}, "nLPHiuxXbTE": {"title": "Trump used DECOY plane to leave NATO summit amid Iranian threats: report", "language": "en", "source_name": "Fox News"}, "yChEzeQsH88": {"title": "EXPLOSIVE tests revealed from Fauci\'s phone: \'DEVASTATING\'", "language": "en", "source_name": "Fox News"}, "NpQvPhb-yiQ": {"title": "Trump STUNS Iran with reparations REVERSAL", "language": "en", "source_name": "Fox News"}, "jiqL7xUd77g": {"title": "\'ANSWER THE QUESTION\': DSA-backed candidate GRILLED on policies", "language": "en", "source_name": "Fox News"}, "wA6X54rXWkE": {"title": "JUST IN: Trump RESPONDS to Iranian compensation demands", "language": "en", "source_name": "Fox News"}, "WpTl6WqDie4": {"title": "Karoline Leavitt: DSA should be called ‘DESTRUCTIVE SOCIALISTS OF AMERICA’", "language": "en", "source_name": "Fox News"}, "1i_mreaCwNo": {"title": "Dan Bongino: This is really a watershed moment", "language": "en", "source_name": "Fox News"}, "xPKEERvzz3Q": {"title": "Jonathan Turley: This is ABSURD", "language": "en", "source_name": "Fox News"}, "1tZM9z7VdNo": {"title": "Ex-Bush speechwriter: This is CRINGEWORTHY", "language": "en", "source_name": "Fox News"}, "J-weWSP5zC4": {"title": "Hannity: THIS is one ‘GIANT POWER GRAB’", "language": "en", "source_name": "Fox News"}, "IyLX3muEQLs": {"title": "Here\'s why Fauci should get IMMUNITY: Paul Mauro", "language": "en", "source_name": "Fox News"}, "cvJsft2UCWw": {"title": "Gutfeld: This makes Jussie Smollett look like ‘Honest Abe’", "language": "en", "source_name": "Fox News"}, "66zrEaAJjQw": {"title": "McEnany: It’s game time…", "language": "en", "source_name": "Fox News"}, "1VC5KUoIBd0": {"title": "McEnany: The Democratic Party is now the DSA party", "language": "en", "source_name": "Fox News"}, "nICLbdy2gNg": {"title": "IRANIAN INFRACTION: Iran under serious economic pressure", "language": "en", "source_name": "Fox News"}, "8LAVdU2b_9k": {"title": "‘The Five’ on the Left’s ‘STUPID IDEA GENERATOR’", "language": "en", "source_name": "Fox News"}, "UTifI5lwLaQ": {"title": "\'The Five\': AOC DISTANCES herself from Francesca Hong", "language": "en", "source_name": "Fox News"}, "9tX7W93xvCo": {"title": "Watch Andy Burnham\'s reaction to hundreds crossing to UK in small boat", "language": "en", "source_name": "ITV News"}, "eFjVHtMk1uQ": {"title": "The maximum-security prison where suspected crime boss Daniel Kinahan is being kept", "language": "en", "source_name": "ITV News"}, "yH62V0dRheU": {"title": "Watch the moment bungalow destroyed in massive explosion near Milton Keynes", "language": "en", "source_name": "ITV News"}, "T8Ys0p-dPzQ": {"title": "Watch as PC Harper\'s mother sends message to Andy Burnham over prisoner release scheme", "language": "en", "source_name": "ITV News"}, "oK_VJtso108": {"title": "We investigated the reality for people working on zero-hour contracts", "language": "en", "source_name": "ITV News"}, "d_82SQA-CDE": {"title": "How a landmark trial of social media company META is unfolding", "language": "en", "source_name": "ITV News"}, "h2MnuGw804s": {"title": "The full timeline: How double murderer Simon Levy was caught after years of crimes", "language": "en", "source_name": "ITV News"}, "nrrXp08sRHg": {"title": "Drought to hit more parts of England as heatwave beckons and water supplies dwindle", "language": "en", "source_name": "ITV News"}, "wH6xf27xUXs": {"title": "Uefa reaffirms World Cup boycott threat after rejecting Infantino apology", "language": "en", "source_name": "ITV News"}, "aw1-d6WAJYQ": {"title": "The threats facing the Welsh language", "language": "en", "source_name": "ITV News"}, "Vizq7-9V95M": {"title": "UN\xa0 group accuses UK of \'psychological torture\' over indefinite jail terms for prisoner", "language": "en", "source_name": "ITV News"}, "IaV0V9keZv8": {"title": "Russian missile and drone strikes bombard Ukraine\'s capital as Zelenskyy pleads for more ammunition", "language": "en", "source_name": "ITV News"}, "xTF5wQcD6tI": {"title": "Former neo-Nazi withdraws Conservative candidacy as Badenoch \'sad\' to see him step down", "language": "en", "source_name": "ITV News"}, "JyP_R7TFKEM": {"title": "Watch how scammers make millions of pounds by staging fake car crashes", "language": "en", "source_name": "ITV News"}, "t5109kT5b-c": {"title": "PC Andrew Harper\'s widow says Burnham \'should be ashamed\' of early release decision", "language": "en", "source_name": "ITV News"}, "9xY0utbU1I8": {"title": "Why the government is finally taking action on this cancer-linked pregnancy pill", "language": "en", "source_name": "ITV News"}, "GAEuBCQcVvE": {"title": "CCTV police believe shows man carrying suitcase \'containing dead Scottish woman\'s body\' in Athens", "language": "en", "source_name": "ITV News"}, "x2ukdPFIJcI": {"title": "PC Harper\'s killers among those who may walk free from jail early after government review", "language": "en", "source_name": "ITV News"}, "eA-jmvzcw6g": {"title": "Danube\'s critically low level reveals relics from the past", "language": "en", "source_name": "PBS NewsHour"}, "-BBRMrAuBgU": {"title": "Family of Marine veteran imprisoned in Russia warns he\'s near death", "language": "en", "source_name": "PBS NewsHour"}, "jy-EHtzb-Ls": {"title": "Tamara Keith and Amy Walter on the energy behind outsider candidates", "language": "en", "source_name": "PBS NewsHour"}, "SWtcb-K63pI": {"title": "What Meta\'s new open-source AI model means for the future of artificial intelligence", "language": "en", "source_name": "PBS NewsHour"}, "cdRooXdFzQA": {"title": "State voting officials join CISA security call with election 85 days away", "language": "en", "source_name": "PBS NewsHour"}, "kFzsQQWb2t4": {"title": "Iran demands U.S. pay for war damage before agreeing to reopen Strait of Hormuz", "language": "en", "source_name": "PBS NewsHour"}, "EBiKASqc2Kg": {"title": "News Wrap: Trump signs order calling for MMR vaccine split into 3 shots", "language": "en", "source_name": "PBS NewsHour"}, "g-seih2uLNQ": {"title": "7.4-magnitude earthquake rocks Colombia, crumbling homes and buildings", "language": "en", "source_name": "PBS NewsHour"}, "Q9URCTFYkMo": {"title": "Brooks and Capehart on what progressive primary wins mean for Democrats", "language": "en", "source_name": "PBS NewsHour"}, "LDQRXWGiFSA": {"title": "How Lahaina is recovering 3 years after the devastating fires", "language": "en", "source_name": "PBS NewsHour"}, "kr7UD2DqonI": {"title": "Colombia\'s new president reflects Latin America\'s right-wing shift", "language": "en", "source_name": "PBS NewsHour"}, "9ZcYkG3eD_w": {"title": "Drought and aging infrastructure plunge Puerto Rico into a water crisis", "language": "en", "source_name": "PBS NewsHour"}, "igs7BN-inVA": {"title": "What\'s holding up the Iran-Oman proposal to reopen the Strait of Hormuz", "language": "en", "source_name": "PBS NewsHour"}, "oSYBveL6J6c": {"title": "Blanche on track for narrow Senate confirmation as attorney general", "language": "en", "source_name": "PBS NewsHour"}, "4c1P1hiBSAc": {"title": "News Wrap: Court rules Trump needs Congress to approve White House ballroom", "language": "en", "source_name": "PBS NewsHour"}, "pYthwnBzk3A": {"title": "What the July jobs report reveals about the strength of the economy", "language": "en", "source_name": "PBS NewsHour"}, "hSW29bbS2aE": {"title": "WATCH: Sen. Cassidy says he\'ll vote to confirm Todd Blanche for attorney general", "language": "en", "source_name": "PBS NewsHour"}, "LzI5l1ZVL8g": {"title": "Exhibition reveals how summers by the sea shaped three great American painters", "language": "en", "source_name": "PBS NewsHour"}, "7USC_IIRJCk": {"title": "How American English evolved from its British roots", "language": "en", "source_name": "PBS NewsHour"}, "L2_6ucG8vtg": {"title": "How the Cockroach Janta Party toppled an Indian government minister | The View From", "language": "hi", "source_name": "The Guardian"}, "JHrboQm4GRw": {"title": "Why so many American Jews are turning away from Israel | On the Ground", "language": "en", "source_name": "The Guardian"}, "6dyW4sk6vS4": {"title": "How Chinese tourism fuels illegal animal trafficking | On the Ground", "language": "en", "source_name": "The Guardian"}, "oEp5a1yvrAI": {"title": "The cartel crisis Mexico wants to hide this World Cup | On The Ground", "language": "en", "source_name": "The Guardian"}, "oP_wtmUDKPM": {"title": "Has Cornwall’s housing crisis created a van life generation? | Focus", "language": "en", "source_name": "The Guardian"}, "12LgcpLtJ0Q": {"title": "Burnham\'s Britain: six days in the place that just changed our politics | Anywhere but Westminster", "language": "en", "source_name": "The Guardian"}, "_u8znNPKfc8": {"title": "Why Americans are injecting themselves with unregulated ‘miracle’ drugs | On the Ground", "language": "en", "source_name": "The Guardian"}, "ljMQCsaPukE": {"title": "Aiden\'s story of life as a young carer | Documentary", "language": "en", "source_name": "The Guardian"}, "Kce0GvSHhRk": {"title": "‘I almost forgot how to date’ | The Global Dating Crisis: episode 3", "language": "ko", "source_name": "The Guardian"}, "aMkZ-yxL5wo": {"title": "Vape shops but no jobs: one young man’s search for work in Grimsby | Focus", "language": "en", "source_name": "The Guardian"}, "QSnH2Z5lGqk": {"title": "Is the far right driving a Christian revival in the UK? | On the Ground", "language": "en", "source_name": "The Guardian"}, "6BrCgcjtzks": {"title": "Why Lebanon is divided over the war with Israel | On The Ground", "language": "ar", "source_name": "The Guardian"}, "HfwQvfchMXU": {"title": "The ‘MeToo’ Backlash | The Global Dating Crisis: episode 2", "language": "en", "source_name": "The Guardian"}, "QYCwAAgBj3Y": {"title": "The Muslim Vote: Democratic threat or Islamophobic myth? | On the Ground", "language": "en", "source_name": "The Guardian"}, "Gw78EcCAjdM": {"title": "The story continues: Points of view, 40 years on", "language": "en", "source_name": "The Guardian"}, "pfEHQH1RSr0": {"title": "Has the manosphere ruined dating? | The Global Dating Crisis: episode 1", "language": "en", "source_name": "The Guardian"}, "ebZyXIhUQKY": {"title": "Colombia earthquake death toll rises above 200 as rescuers race to find survivors", "language": "en", "source_name": "Al Jazeera English"}, "ixiGUAHPx-0": {"title": "Trump claims US \'owns\' Hormuz as Iran lays out conditions to reopen strait", "language": "en", "source_name": "Al Jazeera English"}, "jcH4pfWAASc": {"title": "Western US wildfires force tens of thousands to evacuate", "language": "en", "source_name": "Al Jazeera English"}, "Ko2VBJWMubs": {"title": "South Korean readers mourn novelist Higashino KeigoーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "9ZdQeIWs9U4": {"title": "Sendai: In Deep Green - A Cat\'s-Eye View of Japan", "language": "en", "source_name": "NHK World Japan"}, "cwDdkLhLK0M": {"title": "Mobile clinics deliver peace of mind to hibakushaーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "eATjicMDo8g": {"title": "The story behind Trump\'s Air Force One deception in Turkey", "language": "en", "source_name": "Reuters"}, "srt3j3Wqwoc": {"title": "Wall Street ends down as US-Iran peace optimism fades", "language": "en", "source_name": "Reuters"}, "Vymlju5tHKY": {"title": "Dozens of utility poles block a billion-dong road | News Cluster | VTV24", "language": "vi", "source_name": "VTV24"}, "ibBQyM52ENA": {"title": "Middle East Tensions Continue to Drive Global Oil Markets | International News Highlights | VTV24", "language": "vi", "source_name": "VTV24"}, "Dca6-CmVqwI": {"title": "Plan to cut construction code red tape won\'t have much effect on housing supply: expert | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "D3NZSc3_nQE": {"title": "Middle aged Australians taking drastic measures to crack into housing market | ABC News", "language": "en", "source_name": "ABC News Australia"}, "uMpOcY9jGr4": {"title": "Rescuers race against time to find Colombia quake survivors | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "XAVxqDuzuiM": {"title": "Why is Ebola infecting people at \'unprecedented speed\' in DR Congo and can it be stopped? | DW News", "language": "en", "source_name": "DW News"}, "qLGimXal0ZQ": {"title": "Rescuers race to find survivors after Colombia\'s deadly earthquake | DW News", "language": "en", "source_name": "DW News"}, "Cwch1Z5D9Jo": {"title": "Latest news bulletin | August 12th, 2026 – Morning", "language": "en", "source_name": "Euronews"}, "oyoH_u68aNk": {"title": "Hungary\'s parliament approves Orbán-critic and former judge András Baka as president", "language": "en", "source_name": "Euronews"}, "6QFO8cE7Q84": {"title": "Syria sentences Bashar al-Assad to death in absentia over atrocities • FRANCE 24 English", "language": "en", "source_name": "France 24 English"}, "lSrRGed6rZU": {"title": "Death toll from Colombia quake hits 250 | Morning News Bulletin 12 August 2026", "language": "ko", "source_name": "SBS News"}, "Ejq1J1spm_0": {"title": "RBA holds fire but rate rise still on the table | Oil spikes | Meta\'s new AI model", "language": "ko", "source_name": "SBS News"}, "jx4VmZ66Lb8": {"title": "Colombia earthquake kills over 160 people | Trump vaccine overhaul | North Korea troops for Russia", "language": "ko", "source_name": "SBS News"}, "305JCgLhNgA": {"title": "French Newspapers Fear Google AI Will Hit Web Traffic; Google AI Summaries Spark Media Row | DNA", "language": "en", "source_name": "WION"}, "Efm_kJRZ1wE": {"title": "Mumbai Landslide: 2 Killed, Several Trapped | Trump Says His Plane Faced \'Greater Risk\' | WION", "language": "en", "source_name": "WION"}, "64ds2pkJ_tw": {"title": "[Breaking News] Level 4 Landslide Warning Issued for Chichibu, Saitama – Extreme Caution Advised ...", "language": "ja", "source_name": "ANNnewsCH"}, "FgAHPvqTUhM": {"title": "台風15号お盆休み直撃\u3000花火大会中止に…1500発が幻\u3000ビーチにも打撃\u3000東北各地も影響【羽鳥慎一モーニングショー】(2026年8月12日)", "language": "ja", "source_name": "ANNnewsCH"}, "GTYMdF4ULyY": {"title": "Typhoon No. 15 Impacts Mt. Fuji Climbing on \\"Mountain Day\\": Fierce Winds Prompt Safety Guidance f...", "language": "ja", "source_name": "ANNnewsCH"}, "Rt0ClWRYCLU": {"title": "Unprecedented Reverse-Tracking Typhoon No. 15 Rages Across the Capital Region, Toppled Trees Repo...", "language": "ja", "source_name": "ANNnewsCH"}, "cO4U3uP_ndU": {"title": "Human Rights Groups Sue to Block President Trump’s \\"Birthright Citizenship Restrictions\\" Executiv...", "language": "ja", "source_name": "ANNnewsCH"}, "ij1tv3CJLCE": {"title": "从单点突破到区域协同！沪苏浙皖四方联动，加速打造世界级科创中心 | CCTV「焦点访谈」20260811", "language": "zh", "source_name": "CCTV"}, "sMtzQk1V79U": {"title": "登陆风力强、影响范围广、持续时间长……台风“白海豚”登陆后，有哪些防御重点？| CCTV「焦点访谈」20260810", "language": "zh", "source_name": "CCTV"}, "IMDd8HvWnew": {"title": "Is migration \'weaponized\' in Europe?", "language": "en", "source_name": "CGTN"}, "hukd5ds4HVg": {"title": "Stringer Dispatch: Colombia earthquake devastation captured in drone footage", "language": "en", "source_name": "CGTN"}, "VeP3wZo424I": {"title": "日航機墜落事故からきょうで41年\u3000遺族など110人が慰霊登山\u3000事故当時入社していた日航社員は今年度中にゼロに（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "MQNr1Ln_zjM": {"title": "台風17号発生、関東に影響は？海のレジャーは要注意\u3000台風15号が上陸したばかりの関東や東北に接近する可能性（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "Hk2yo4lB18g": {"title": "“異例の逆走ルート”台風15号は熱帯低気圧に\u3000今夜にかけ東京でも“滝のような雨”か（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "0d5qHmdPz_A": {"title": "花火大会で火の粉降りかかり親子やけど\u3000周辺の観客も髪の毛や洋服焦げる被害\u3000主催者「対策取っていたが…」\u3000静岡・伊東市（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "lk0jkldghgQ": {"title": "アスベスト含む糸くずが落下\u3000半蔵門線の一部で始発から約14時間運休\u3000東京メトロが全線で緊急点検（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "gySyDYx9sjI": {"title": "WHO Defends U.S. Childhood Vaccination Schedule Against Trump Executive Order", "language": "vi", "source_name": "HTV Tin Tuc"}, "SrNHdIDzABo": {"title": "6 AM Firefighting and Rescue Drill at Saigon Centre - Tan", "language": "vi", "source_name": "HTV Tin Tuc"}, "B7KyCQ6cHCU": {"title": "Colombia ravaged by 7.4 magnitude earthquake", "language": "en", "source_name": "South China Morning Post"}, "Ved2yJZ5SNg": {"title": "Manulife Hong Kong CEO Wilton Kee runs ahead of risk with preventive care", "language": "en", "source_name": "South China Morning Post"}, "2XE2yh-OzJk": {"title": "MORNING NEWS: Suspect charged for child abuse at Bau Pagoda, Phu Tho | VTV24", "language": "vi", "source_name": "VTV24"}, "f2QSrHXxB4E": {"title": "[Uncut] North Korea launches ballistic missile, no reports of damage, Defense Minister Koizumi lo...", "language": "ja", "source_name": "Kyodo News"}, "GpbMKoQeHms": {"title": "[Breaking] North Korea Launches Ballistic Missile; No Reports of Damage; Defense Minister Koizumi...", "language": "ja", "source_name": "Kyodo News"}, "LJpmYKbpqLo": {"title": "【速報】遺族ら「御巣鷹の尾根」を慰霊登山 日航機墜落41年、520人悼み安全願う", "language": "ja", "source_name": "Kyodo News"}, "38vx1yDKVYs": {"title": "JAL Flight Crash Victims Mourned with Lanterns at Kanna River near Gunma Crash Site", "language": "ja", "source_name": "Kyodo News"}, "pY9E2eiUruQ": {"title": "[Typhoon Information] New Typhoon No. 17 Forms: May Move North and Approach Honshu; Typhoon No. 1...", "language": "ja", "source_name": "TBS News Dig"}, "1OAQJT1jkVk": {"title": "Trump on 3rd term: \\"The law is strict,\\" expresses negative view on running｜TBS NEWS DIG", "language": "ja", "source_name": "TBS News Dig"}, "HJ7tC0cBCNo": {"title": "[Exclusive] Police Stations Attacked with Stolen Motorbike? Two Boys from Adachi Ward Arrested fo...", "language": "ja", "source_name": "TBS News Dig"}, "F9Zelwim9B0": {"title": "実は古いほどおいしい？糖度42度！真夏のサツマイモが甘いワケ\u3000熟成不要の“奇跡”の品種も誕生【Nスタ】｜TBS\xa0NEWS\xa0DIG", "language": "ja", "source_name": "TBS News Dig"}, "QKG3BtuO-r4": {"title": "[81 Years After the War] \\"All Japanese children will be killed...\\" The hardships of second-genera...", "language": "ja", "source_name": "TBS News Dig"}, "NcVbh5WZ1T4": {"title": "从室内剧场到水上舞台，“流动的中国”让这个暑假活力无限！| CCTV「焦点访谈」20260809", "language": "zh", "source_name": "CCTV"}, "7LF0Vva1j54": {"title": "村超、村BA等赛事火爆，带来的流量如何为城市发展经济增量？| CCTV「焦点访谈」20260807", "language": "zh", "source_name": "CCTV"}, "7SWqQpLTaF4": {"title": "Inside El Nino: Ethiopia\'s unpredictable rainfall", "language": "en", "source_name": "CGTN"}, "ZVpkLpSbD6g": {"title": "China designates 16 cities as pilot hubs for \'film+\' model", "language": "en", "source_name": "CGTN"}, "Uhi5jxvVqCA": {"title": "Kam Grand Choirs: A melody of life, nature and unity", "language": "en", "source_name": "CGTN"}, "aIA6RE5e6_c": {"title": "Stringer Dispatch: Deadly wildfires force 20,000 to flee in Canada", "language": "en", "source_name": "CGTN"}, "DQ8wJOsbzO8": {"title": "How Putonghua became ‘common speech’ in China", "language": "en", "source_name": "South China Morning Post"}, "8iLbCCF9vro": {"title": "How the next generation at the family-run ‘White Flower Oil’ is shaping its future", "language": "en", "source_name": "South China Morning Post"}, "2cEyV2et-v4": {"title": "Giorgio Chiellini wants to close the gap between Serie A and the Premier League", "language": "ko", "source_name": "SBS News"}, "qilZIE1yIe4": {"title": "Ex-prime minister warns AUKUS deal may not deliver", "language": "ko", "source_name": "SBS News"}, "oJ0TlCn-_WY": {"title": "Baker Boy sweeps the 2026 National Indigenous Music Awards", "language": "ko", "source_name": "SBS News"}, "NKVCFfNQkXI": {"title": "Superannuation death benefits | Rental property bonds | Australian Census-eve", "language": "ko", "source_name": "SBS News"}, "0F3GjRYmNRE": {"title": "Israel rejects US led Gaza plan | Trump opts for \\"low key” negotiations with Iran to end the war", "language": "ko", "source_name": "SBS News"}, "e4tOzuoE_ns": {"title": "Bankruptcy of Confectionery Shops Like Cake and Wagashi Stores at Record Pace Amid Weak Yen and D...", "language": "ja", "source_name": "ANNnewsCH"}, "wozNzJhlMrI": {"title": "[Breaking News] Level 4 Storm Surge Warning Issued for Kagawa Prefecture; High Alert for Flooding...", "language": "ja", "source_name": "ANNnewsCH"}, "gXjq6FGMb1o": {"title": "\\"Overflowing Karaage Bento\\" Shop Reopens, Bringing Smiles to Residents Two Weeks After Kumamoto E...", "language": "ja", "source_name": "ANNnewsCH"}, "ZSh9tXaejC8": {"title": "コロンビアでマグニチュード7.4の地震\u3000死者少なくとも254人\u30004000人近くが行方不明の恐れ\u3000停電や断水などインフラ被害続く（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "7GP-t3_tU70": {"title": "最高高度約90km・約690km飛翔、日本のEEZ外側の日本海に落下か\u3000北朝鮮の弾道ミサイル発射に小泉防衛大臣「厳重に抗議」（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "yLbXLtDsUuU": {"title": "台風15号\u3000観測史上初の茨城県上陸\u3000列島横断中\u3000関東で3人けが強風で倒木など被害相次ぐ（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "kgpB0vNMkB8": {"title": "アサド前大統領に死刑判決\u3000内戦中の計画的殺人や拷問などの罪\u3000アサド氏はロシア亡命で刑の執行は不透明（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "mdltHbBwD9g": {"title": "熊本地震発生から2週間\u3000仮設住宅の整備進む\u3000避難所に約3700人（2026年08月12日）", "language": "ja", "source_name": "FNN News"}, "44WL7zEIQKw": {"title": "Words a mother left behind: Nagasaki legacyーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "yYSYeEwB1hk": {"title": "Thai school shooting leaves 6 dead, including gunmanーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "iKSulOW9Yp4": {"title": "Japanese schools rethink summer homeworkーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "Cg-BLgxVifU": {"title": "Japanese consumers to feel more heat from producer price riseーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "QFGad6OuYGo": {"title": "Myanmar\'s Min Aung Hlaing visiting ThailandーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "JYofrXryEno": {"title": "Archiving the struggles of atomic bomb survivorsーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "j5z7DxT1sYs": {"title": "【速報】台風15号、関東に上陸の恐れ 茨城・大洗の海岸に高波", "language": "ja", "source_name": "Kyodo News"}, "q9R_eAzwdtA": {"title": "Michibiki-7 Launch Successful: Advancing Toward Japan\'s Version of GPS with the H3 Rocket", "language": "ja", "source_name": "Kyodo News"}, "YwfD_qigqmQ": {"title": "[Breaking News] Michibiki-7 successfully launched by JAXA on H3 rocket", "language": "ja", "source_name": "Kyodo News"}, "MXrsKxr9CSI": {"title": "一度は沈んだジュエリーの“定番”4℃が復活!!\u3000売上高は前期比52%増、原動力はジュエリーではなく高級腕時計？【モヤモヤ経済クリアファイル】", "language": "ja", "source_name": "Kyodo News"}, "a6LclURjBM0": {"title": "[Breaking] Fukuoka Prefectural Assembly Chairman Kurauchi denies anonymous testimony: \\"A misunder...", "language": "ja", "source_name": "Kyodo News"}, "qBiOXO6gmEM": {"title": "\\"Able to go to work for the first time since the earthquake\\": Yatsushiro City opens \\"safe space f...", "language": "ja", "source_name": "TBS News Dig"}, "So8ffMng2dA": {"title": "\\"To Ensure Thorough Heatstroke Prevention\\": Tokyo Fire Department Introduces Fan-Equipped Vests a...", "language": "ja", "source_name": "TBS News Dig"}, "D6NN1uPQ75A": {"title": "\\"Houthi rebels claim attack on ship carrying Saudi military equipment\\": Cargo ship and others att...", "language": "ja", "source_name": "TBS News Dig"}, "1eOCBUFWFNw": {"title": "Typhoon No. 15 Downgraded to Tropical Depression, Yet Risk of Heavy Rain Remains from Tohoku to K...", "language": "ja", "source_name": "TBS News Dig"}, "ouCWahyuvhY": {"title": "Shaka Nyorai statue damaged again by Kumamoto earthquake just months after March restoration; \\"we...", "language": "ja", "source_name": "TBS News Dig"}, "7jGPKk3hTyo": {"title": "Syria sentences Bashar al-Assad and former officials to death over war crimes", "language": "en", "source_name": "Al Jazeera English"}, "y_vOaUjcqf4": {"title": "Trump should ‘take the less bad deal’ and accept Iranian sovereignty over Hormuz: Analysis", "language": "en", "source_name": "Al Jazeera English"}, "p17mIv6BnuU": {"title": "Syrians celebrate death sentences for Bashar al-Assad and senior officials", "language": "en", "source_name": "Al Jazeera English"}, "Ic-FXeUtjjQ": {"title": "Hungary Elects Ousted Judge Andras Baka as New President", "language": "en", "source_name": "Al Jazeera English"}, "qX1UEUuSQds": {"title": "Israeli Settlers and Soldiers Tighten Grip on West Bank Villages", "language": "en", "source_name": "Al Jazeera English"}, "gU9B1rnRL5c": {"title": "Israeli settler attacks in occupied West Bank reach record levels in 2026", "language": "en", "source_name": "Al Jazeera English"}, "nnPPV5v6zl0": {"title": "Iran–US talks in Limbo as Pakistan and Oman mediate", "language": "en", "source_name": "Al Jazeera English"}, "IX6GDdiKafU": {"title": "“白海豚”停编 这个台风为何“不走寻常路”？| CCTV「东方时空」", "language": "zh", "source_name": "CCTV"}, "qG97ZIHYSfY": {"title": "走访重大工程建设一线！三峡新通道建设进度如何？光伏发电如何应对一朵乌云？| CCTV「焦点访谈」20260806", "language": "zh", "source_name": "CCTV"}, "3o7vwhQe0XE": {"title": "Stringer Dispatch: Colombia\'s strongest earthquake this century kills at least 132", "language": "en", "source_name": "CGTN"}, "QO31e56Iw38": {"title": "Tech tourism gains popularity among international visitors to China", "language": "en", "source_name": "CGTN"}, "dA-BJFC8x2w": {"title": "US water systems under cyber attack | DW News", "language": "en", "source_name": "DW News"}, "b31YP1yrAZQ": {"title": "North Korea expands its role in Russia\'s war against Ukraine | DW News", "language": "en", "source_name": "DW News"}, "RSFq2lpou0U": {"title": "Doctors warn Trump’s vaccine plan could endanger children | DW News", "language": "en", "source_name": "DW News"}, "ItbDPmNvQ80": {"title": "Hungary elects Orban critic as president. Can it roll back 16 years of Orban-era rule? | DW News", "language": "en", "source_name": "DW News"}, "Y0FBk17C8R4": {"title": "Millions head to Spain for once-in-a-lifetime solar eclipse | DW News", "language": "en", "source_name": "DW News"}, "NM-JQoQbAeM": {"title": "Khamenei\'s new war chiefs: Preparing Iran for a long fight? | DW News", "language": "en", "source_name": "DW News"}, "oh6AOiXkWfc": {"title": "Qatar sees \'positive\' signs in Iran-Oman talks on reopening Hormuz", "language": "en", "source_name": "Euronews"}, "sbPmUv-KtWo": {"title": "UK set for \'warmest summer on record\' as Europe preps for yet another heatwave", "language": "en", "source_name": "Euronews"}, "FKS9FmqE1WQ": {"title": "Latest news bulletin | August 11th, 2026 – Evening", "language": "en", "source_name": "Euronews"}, "QGX-cK2Leoc": {"title": "Grape harvest in Hungary\'s Villány wine region down 30% due to drought", "language": "en", "source_name": "Euronews"}, "UM6rAFVg5EE": {"title": "Death toll from Colombia earthquake continues to rise • FRANCE 24 English", "language": "en", "source_name": "France 24 English"}, "_z7LnREq7IU": {"title": "Ukraine says Russia fired North Korean missiles • FRANCE 24 English", "language": "en", "source_name": "France 24 English"}, "wZnyt7wSlRg": {"title": "Europe battles wildfires and braces for another summer heatwave • FRANCE 24 English", "language": "en", "source_name": "France 24 English"}, "UboavSKpns8": {"title": "Colombia rescuers race against the clock to search for earthquake survivors • FRANCE 24 English", "language": "en", "source_name": "France 24 English"}, "LiJfujkCeRY": {"title": "Why some Australians could go uncounted in this year\'s census", "language": "ko", "source_name": "SBS News"}, "z-AfkbcApH4": {"title": "Iran\'s fresh demands over Strait of Hormuz negotiations | Ukraine agreement on US missile shipments", "language": "ko", "source_name": "SBS News"}, "UVP8SvF1gPg": {"title": "Saudi Arabia, Turkey and Pakistan sign historic defence pact amid Iran tensions", "language": "ko", "source_name": "SBS News"}, "SQzNBQ4N7Pw": {"title": "11:30 IRAN CRISIS: YEMEN SHIPPING STILL", "language": "vi", "source_name": "HTV Tin Tuc"}, "xMeycuVWXPA": {"title": "11:30 AM Launch of the Vegetarian Culinary Association", "language": "vi", "source_name": "HTV Tin Tuc"}, "I88Zg-08lGM": {"title": "11:30 AM Rescuers in Colombia race against time", "language": "vi", "source_name": "HTV Tin Tuc"}, "G8N2udMAM1Q": {"title": "11:30 AM UKRAINE CRISIS ZELENSKIY", "language": "vi", "source_name": "HTV Tin Tuc"}, "XqoIJuMW144": {"title": "11:30 AM NORTH KOREA MISSILES", "language": "vi", "source_name": "HTV Tin Tuc"}, "3X5BPolLmBA": {"title": "11:30 AM Running Man Vietnam Sets a New Record", "language": "vi", "source_name": "HTV Tin Tuc"}, "k9gC8cdAvmw": {"title": "TIN NÓNG 10 PHÚT: Sân bay Tân Sơn Nhất 2 lần tạm dừng cất, hạ cánh vì drone | VTV24", "language": "vi", "source_name": "VTV24"}, "eqw2AJ0V6kQ": {"title": "REAL OR FAKE, VERIFIED: Beware of fake study abroad scholarships | VTV24", "language": "vi", "source_name": "VTV24"}, "g7f4nnPhGVM": {"title": "Danger Warning: Risks of Children Overusing Electronic Devices | News Roundup | VTV24", "language": "vi", "source_name": "VTV24"}, "nSBBMYguGGE": {"title": "Investigating Child Abuse Clip at Bau Pagoda, Vinh Phuc | News Roundup | VTV24", "language": "vi", "source_name": "VTV24"}, "vVQQq3AOGkA": {"title": "Russia says hundreds of Ukrainian drones shot down as attacks hit deep inside its territory", "language": "en", "source_name": "Al Jazeera English"}, "g3Bt3LGxOsc": {"title": "Rwanda\'s AI-powered healthcare helps community workers treat more patients", "language": "en", "source_name": "Al Jazeera English"}, "sRCtc32P2Hs": {"title": "Ukraine strikes Russia : Drones hit Wildberries warehouse and oil refinery", "language": "en", "source_name": "Al Jazeera English"}, "f0taseAkoGc": {"title": "Mount Bromo in Flames: East Java Wildfires Threaten Livelihoods", "language": "en", "source_name": "Al Jazeera English"}, "XKley_Z3ZzU": {"title": "Under Fire: Lebanon’s first responders | Al Jazeera Originals", "language": "en", "source_name": "Al Jazeera English"}, "GcDgbpYNAJY": {"title": "Sednaya prison survivor recounts torture under Bashar al Assad’s rule", "language": "en", "source_name": "Al Jazeera English"}, "HEnM7F91Pi8": {"title": "Filipino seafarers return to sea despite war risks in the Strait of Hormuz and Black Sea", "language": "en", "source_name": "Al Jazeera English"}, "J5oT5rAerBk": {"title": "Colombia earthquake: At least 132 people have been confirmed dead", "language": "en", "source_name": "Al Jazeera English"}, "5D5s7zPI1No": {"title": "Atef Najib embodied everything Syrians hated about Assad\'s regime: Analysis", "language": "en", "source_name": "Al Jazeera English"}, "AtDXBWMbJtI": {"title": "Qatar says Iran-Oman talks on Hormuz reached advanced stage", "language": "en", "source_name": "Al Jazeera English"}, "jK7ueKiAYxU": {"title": "开设首店、新品首发、首展、首秀、首演……“首发经济”第一次纳入国家五年发展规划 | CCTV「焦点访谈」20260805", "language": "zh", "source_name": "CCTV"}, "O34mNBDyYPE": {"title": "碳纤维、芯片领域“卡脖子”关键技术如何攻破？记者探访工厂科研一线 | CCTV「焦点访谈」20260804", "language": "zh", "source_name": "CCTV"}, "wuQ26k-Lnh4": {"title": "“十五五”开局之年行程过半，重大工程建设有何新进展？中国消费又是如何向“新”而行？| CCTV「焦点访谈」20260803", "language": "zh", "source_name": "CCTV"}, "J1Ft0kW5r2g": {"title": "Ministry of Construction explains social housing purchase prices | News Roundup | VTV24", "language": "vi", "source_name": "VTV24"}, "xlJojLy1XwM": {"title": "Russia-Ukraine Intensify Attacks on Multiple Fronts | International News Update | VTV24", "language": "vi", "source_name": "VTV24"}, "pyzGJe34pPI": {"title": "Earthquake Disaster in Colombia | VTV24", "language": "vi", "source_name": "VTV24"}, "_J2Hrj4FXZo": {"title": "BREAKING NEWS | 188 still missing after strongest earthquake in a decade in Colombia | VTV24", "language": "vi", "source_name": "VTV24"}, "XR0DyCZ8qis": {"title": "Wildfires are \\"spreading\\" to the European economy | VTV24", "language": "vi", "source_name": "VTV24"}, "TVI3Gyi6hNU": {"title": "World Panorama August 11: Colombia declares national disaster following earthquake | VTV24", "language": "vi", "source_name": "VTV24"}, "xOUElf7catg": {"title": "Is a university degree still a wise investment? | DW News", "language": "en", "source_name": "DW News"}, "w5MlP3Eijls": {"title": "Putin\'s ban provokes protest | DW News", "language": "en", "source_name": "DW News"}, "gKwEC4xfzyU": {"title": "Deadly earthquake hits Colombia weeks after Venezuela quakes killed thousands | DW News", "language": "en", "source_name": "DW News"}, "_bfKjtHhuNY": {"title": "Russian court excludes anti-war party Yabloko from September election | DW News", "language": "en", "source_name": "DW News"}, "P3W6dKx7u1Y": {"title": "Chinese robot maker Unitree set to become China\'s first humanoid robot stock | DW News", "language": "en", "source_name": "DW News"}, "KojkzCD-vbI": {"title": "Spain plans to relocate 1,300 migrant children from overcrowded Ceuta | DW News", "language": "en", "source_name": "DW News"}, "Pbj1dSBuizE": {"title": "Ukrainian drone attack on oil hub deep inside Russia kills 13 | DW News", "language": "en", "source_name": "DW News"}, "iBOLgaOaMqg": {"title": "Indonesia\'s boom is under threat. Here’s why... | DW News", "language": "en", "source_name": "DW News"}, "SDG2tsdrNXQ": {"title": "BREAKING: Deadly earthquake leaves trail of destruction across Colombia | DW News", "language": "en", "source_name": "DW News"}, "SWwG35EN3uE": {"title": "Ebola death toll in DR Congo outbreak rises to more than 2,000, officials say", "language": "en", "source_name": "Euronews"}, "wZQCTFCbguI": {"title": "Syria sentences Bashar al-Assad to death in absentia for war crimes and crimes against humanity", "language": "en", "source_name": "Euronews"}, "lppa_yDRI3o": {"title": "Latest news bulletin | August 11th, 2026 – Midday", "language": "en", "source_name": "Euronews"}, "TE_2lS7aGGw": {"title": "Russia fired North Korean missiles in attack that killed six in Zaporizhzhia, Zelenskyy says", "language": "en", "source_name": "Euronews"}, "96XSuCOezRA": {"title": "At least 240 killed in Colombia\'s biggest earthquake in years", "language": "en", "source_name": "Euronews"}, "j09oMrfkvlM": {"title": "Trump signs order calling for fewer childhood vaccines, going against medical groups’ advice", "language": "en", "source_name": "Euronews"}, "DweMoLtcFHM": {"title": "Trump boarded secret flight via catering truck due to Iran threat, US media reports • FRANCE 24", "language": "en", "source_name": "France 24 English"}, "PFaSNToDmvw": {"title": "Jury selection begins for ex-gang leader charged \u200bwith Tupac Shakur\'s 1996 murder • FRANCE 24", "language": "en", "source_name": "France 24 English"}, "85Uk6EN2QwM": {"title": "Donald Trump order reduces US vaccine schedule • FRANCE 24 English", "language": "en", "source_name": "France 24 English"}, "BfcPG9CHy1I": {"title": "Esports World Cup underway in Paris • FRANCE 24 English", "language": "en", "source_name": "France 24 English"}, "dYT3nQx-q-Q": {"title": "UEFA, CONCACAF and AFC accuse Gianni Infantino of \'fundamental breach of trust\' • FRANCE 24", "language": "en", "source_name": "France 24 English"}, "EmgkCFz6hxw": {"title": "Kara Robinson Chamberlain was kidnapped at 15, now she investigates other true crime cases", "language": "en", "source_name": "ABC News"}, "_rI35zP5hkM": {"title": "Baby pulled alive from rubble as Colombia earthquake death toll rises", "language": "en", "source_name": "ABC News"}, "dZl48_OccxI": {"title": "Judge rules jurors in Mangione murder trial will remain anonymous", "language": "en", "source_name": "ABC News"}, "VSNnLqnKn30": {"title": "Trump\'s secret flight out of Turkey amid possible Iranian assassination threat", "language": "en", "source_name": "ABC News"}, "_W2-6brRoxc": {"title": "U.S. Marine veteran freed from Russian prison", "language": "en", "source_name": "ABC News"}, "xtK9m2CRTwA": {"title": "Lindsay Clancy murder trial latest: Psychiatrist takes the stand", "language": "en", "source_name": "ABC News"}, "5DrL8-zGs5I": {"title": "What Colombia\'s earthquake means for its fight against rebels and gangs | Global News Podcast", "language": "en", "source_name": "BBC News"}, "BJS_4C8a9WQ": {"title": "Desperate search for survivors as 240 confirmed dead in Colombia earthquake | BBC News", "language": "en", "source_name": "BBC News"}, "Az9rj1j5OAs": {"title": "Iran-backed Houthis launch deadly attack on ship in Red Sea | BBC News", "language": "en", "source_name": "BBC News"}, "ngMlRedHkTQ": {"title": "How Donald Trump swapped planes to avoid threat from Iran | BBC News", "language": "en", "source_name": "BBC News"}, "5CmRDWz7su4": {"title": "Jewish headteachers in England \'spend more time on security than teaching\', review finds | BBC News", "language": "en", "source_name": "BBC News"}, "cHCt1bCujp8": {"title": "Europe awaits first total solar eclipse since 1999 | BBC News", "language": "en", "source_name": "BBC News"}, "LSbI4eMnM4c": {"title": "Colombia earthquake death toll rises above 220 | BBC News", "language": "en", "source_name": "BBC News"}, "h7E1i3kQuKM": {"title": "How Russia seizes homes in occupied Ukraine | Global News Podcast", "language": "en", "source_name": "BBC News"}, "QttgvkZvEqI": {"title": "Punk music turns 50 in the UK with festival celebration | BBC News", "language": "en", "source_name": "BBC News"}, "RgMIwOr0Da0": {"title": "Is Modi \'weakened\' by India\'s cockroach protests? | BBC News", "language": "en", "source_name": "BBC News"}, "MgybjaHp0pQ": {"title": "Missile threat against Air Force One prompted Trump\'s plane switch: Sources", "language": "en", "source_name": "CNN"}, "otOpiyRVUBM": {"title": "Colombia earthquake death toll +200 as rescue efforts intensify", "language": "en", "source_name": "CNN"}, "Khl7diwQRA4": {"title": "Iran’s refreshed military, Trump’s plane tactic & more | Military experts on Iran", "language": "en", "source_name": "CNN"}, "bPlcckdGX8s": {"title": "Security experts react to Trump\'s secret flight after Iran threat", "language": "en", "source_name": "CNN"}, "G8F7kiPGcTw": {"title": "Harry Enten shares \'shocking\' finding in latest polling data on Iran war", "language": "en", "source_name": "CNN"}, "_njrvlzYNis": {"title": "Saudi Arabia, Turkey & Pakistan sign NATO-style defense pact", "language": "en", "source_name": "CNN"}, "zD9CqZJtQA8": {"title": "\'This is the future for Europe\': Fareed reacts to Moroccan migrant crisis", "language": "en", "source_name": "CNN"}, "mfxYtpsI7YU": {"title": "Trump is unlawfully constructing White House ballroom: Appeals court", "language": "en", "source_name": "CNN"}, "wJyOP2sedOo": {"title": "Blanche\'s AG confirmation chances resuscitated by key GOP senator", "language": "en", "source_name": "CNN"}, "-9bYuGYSMdE": {"title": "Sen. Amy Klobuchar to face Lisa Demuth in Minnesota gubernatorial general election", "language": "en", "source_name": "NBC News"}, "-tZucq8BUfc": {"title": "Larson, Bronin speak to voters after Bronin’s win in Connecticut House race", "language": "en", "source_name": "NBC News"}, "Q8PiRedE4mo": {"title": "Norman, Graham advance to runoff in South Carolina Republican Senate primary", "language": "en", "source_name": "NBC News"}, "1BMiheFPtEE": {"title": "Norman, Graham advance to South Carolina GOP Senate runoff", "language": "en", "source_name": "NBC News"}, "p5WK5EdMpWs": {"title": "\xa0Sen. Darline Graham advances to runoff in S.C. GOP Senate primary\xa0", "language": "en", "source_name": "NBC News"}, "lX6TzGL9xFA": {"title": "Desperate search for survivors after deadly Colombia earthquake", "language": "en", "source_name": "NBC News"}, "tgosiTYv5uA": {"title": "New bodycam of missing tubers rescue", "language": "en", "source_name": "NBC News"}, "F3XjxTEg50s": {"title": "In her words: Hibakusha preserves her story in EnglishーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "jwsOyK3VzjQ": {"title": "Japan soy sauce maker aims for the worldーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "vfzOogFl8QU": {"title": "Sri Lanka hit by deadly floods, landslidesーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "TKNikTqCLZk": {"title": "Balancing AI efficiency with human expertiseーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "7hv2ss8-co0": {"title": "Symbolic, in sync: Bessent hails yen-boosting joint action between US, JapanーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "tEodeGZ9RfE": {"title": "Japan\'s drone defense challengeーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "GZF2dZJ_wiI": {"title": "UN welcomes Suu Kyi meeting with Red CrossーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "-SPL74VZBoo": {"title": "Let Your Music Be Heard - Chatroom Japan", "language": "en", "source_name": "NHK World Japan"}, "ropV2gtVKKI": {"title": "Trump reportedly smuggled off Air Force One in a catering truck after Iran \'threat\'", "language": "en", "source_name": "Sky News"}, "t-FuefM3daQ": {"title": "QUICK TAKE: Did Trump\'s \\"decoy\\" flight endanger journalists?", "language": "en", "source_name": "Sky News"}, "OXy_MEMTYF0": {"title": "Will Ukraine be able to cope against Russia’s ballistic missile production?", "language": "en", "source_name": "Sky News"}, "5I_x60KKG4M": {"title": "Mother of PC Harper speaks to Sky News about proposed early release of her son\'s killers", "language": "en", "source_name": "Sky News"}, "GI0Ya70xvJg": {"title": "FIFA accuses critics of ‘ongoing effort’ to undermine Gianni Infantino", "language": "en", "source_name": "Sky News"}, "9uV5292kkEA": {"title": "Thousands evacuated as huge wildfire spreads in Canada", "language": "en", "source_name": "Sky News"}, "3NsZJG_8MWg": {"title": "How the war is being brought to Russian consumers", "language": "en", "source_name": "Sky News"}, "xGXBdkVKKeM": {"title": "Europe to experience total solar eclipse on Wednesday while 25 U.S. states will see partial eclipse", "language": "en", "source_name": "CBS News"}, "Uw-okDhmq-o": {"title": "Former Secret Service special agent weighs in on Trump\'s secret plane switch", "language": "en", "source_name": "CBS News"}, "fPdvp7flB6o": {"title": "Doctor points to numerous studies showing COVID vaccine is safe for pregnant women", "language": "en", "source_name": "CBS News"}, "iOmaO3FUIFA": {"title": "New forecast shifts GOP Rep. Max Miller\'s reelection race to \\"toss-up\\"", "language": "en", "source_name": "CBS News"}, "qK2RuuU4cxo": {"title": "On the ground in South Carolina, Wisconsin and Minnesota as primaries near end", "language": "en", "source_name": "CBS News"}, "3NBC9_okFjc": {"title": "Assad sentenced to death in Syria - will he be brought to justice?", "language": "en", "source_name": "Channel 4 News"}, "TCqAlw1uTQc": {"title": "Colombia earthquake: Death toll climbs with more feared trapped", "language": "en", "source_name": "Channel 4 News"}, "owq3d2QV5jI": {"title": "PC Harper’s mum has ‘hope’ Burnham can stop early release of his killers", "language": "en", "source_name": "Channel 4 News"}, "etehHIUl1xc": {"title": "New parliamentary investigation launched against Reform UK", "language": "en", "source_name": "Channel 4 News"}, "OBjCFEhwKU0": {"title": "The shocking reality of Ukraine\'s drone war in an underground hospital", "language": "en", "source_name": "Channel 4 News"}, "7YD-Rew6pDI": {"title": "Two firefighting helicopters collide while tackling wildfires near Athens", "language": "en", "source_name": "Channel 4 News"}, "DH5GqQwCY0A": {"title": "Why has Trump paused new Iran attacks?", "language": "en", "source_name": "Channel 4 News"}, "_Dw8sA4P5SQ": {"title": "Mother of soldier killed in Iranian attack: He was passionate about life", "language": "en", "source_name": "Fox News"}, "1H-6hwKU4dw": {"title": "Tyrus says the WNBA should ‘fund themselves’", "language": "en", "source_name": "Fox News"}, "xqrE1GKFVOI": {"title": "DODGING CROSSHAIRS: Reports reveal Trump\'s decoy in evading Iran\'s threat", "language": "en", "source_name": "Fox News"}, "GETEVaODBig": {"title": "BREAKING: Darline Graham advances to South Carolina GOP US Senate runoff", "language": "en", "source_name": "Fox News"}, "ZR8r6Vll4Js": {"title": "Grieving partner of woman killed by dangerous driver adds to criticism of early prison releases", "language": "en", "source_name": "ITV News"}, "FIlNzVs8nLo": {"title": "Fires burn across the country as the UK faces it\'s hottest summer on record", "language": "en", "source_name": "ITV News"}, "uBAH0zCs1jU": {"title": "Colombia earthquake: Watch as survivors are pulled from damaged buildings", "language": "en", "source_name": "ITV News"}, "r_-EaMEGpoA": {"title": "England drought: Boaters race to escape canal closures as water levels run low", "language": "en", "source_name": "ITV News"}, "up5b5nfbGLw": {"title": "Thunderstorms to hit UK with potential flash flooding: Who will be impacted?", "language": "en", "source_name": "ITV News"}, "piCmKyssDWI": {"title": "WATCH: Trump arrives at finale of Patriot Games high school athlete competition", "language": "en", "source_name": "PBS NewsHour"}, "uhnUxu-zZLs": {"title": "WATCH: Border czar Homan says ICE is still making record arrests \'despite the radical courts\'", "language": "en", "source_name": "PBS NewsHour"}, "WFD1IIL1huM": {"title": "What to know as your driver\'s license gets a digital upgrade", "language": "en", "source_name": "PBS NewsHour"}, "JRXGlR8Cs2I": {"title": "Top adviser to Iran\'s Revolutionary Guard leader sits down for rare interview", "language": "en", "source_name": "PBS NewsHour"}, "a7co1CeEq44": {"title": "News Wrap: Syria sentences Bashar al-Assad to death in absentia", "language": "en", "source_name": "PBS NewsHour"}, "5s2YlMH5FN8": {"title": "How a catering truck helped Trump secretly switch planes amid threat from Iran", "language": "en", "source_name": "PBS NewsHour"}, "zoYrLNQsnVM": {"title": "US wildfires: Tens of thousands evacuated in Washington state", "language": "en", "source_name": "Al Jazeera English"}, "KINuNYV9Fc4": {"title": "Syria verdict: Atef Najib accused of overseeing 2011 crackdown", "language": "en", "source_name": "Al Jazeera English"}, "Ty1VeJe9bXc": {"title": "Dozens of settlers escorted by Israeli forces have attacked Palestinians north of Ramallah", "language": "en", "source_name": "Al Jazeera English"}, "kWfWEpeZy4k": {"title": "Sudan authorities accuse RSF of killing 25 civilians found in mass grave", "language": "en", "source_name": "Al Jazeera English"}, "kEPNNbVacW4": {"title": "Ofer Cassif: Attacks against Palestinians in the West Bank amount to \'state terrorism\'", "language": "en", "source_name": "Al Jazeera English"}, "IJKPpIbFjzU": {"title": "Iran\'s new military appointments signal offensive strategy: Analysis", "language": "en", "source_name": "Al Jazeera English"}, "Yv9JDpD3Vl0": {"title": "Trump demands compensation from Iran, claims US control of Strait of Hormuz", "language": "en", "source_name": "Al Jazeera English"}, "9CQUQTrxaRM": {"title": "Ariana Grande: why we can’t stop talking about women\'s bodies | The Global Story", "language": "en", "source_name": "BBC News"}, "wiNokGAVXPU": {"title": "How one of the most “dangerous and predatory” men was left free to kill | BBC News", "language": "en", "source_name": "BBC News"}, "YyfKXJsOGR8": {"title": "Several dead and injured after Thai teen shoots grandparents and schoolteachers | BBC News", "language": "en", "source_name": "BBC News"}, "EhV6RK9mdIA": {"title": "Thailand school shooting leaves at least six dead and 15 injured | BBC News", "language": "en", "source_name": "BBC News"}, "sw_PRD8ReHc": {"title": "Iran says deal with Oman on Strait of Hormuz is in final stages | BBC News", "language": "en", "source_name": "BBC News"}, "xvgqsImOsGI": {"title": "How progressive Abdul El-Sayed won in Michigan - and why it could shake-up US politics | BBC News", "language": "en", "source_name": "BBC News"}, "yFxq1i_KOC4": {"title": "Why Israeli troops in Lebanon are not withdrawing | Global News Podcast", "language": "en", "source_name": "BBC News"}, "trQbTazXW9g": {"title": "Teen kills students & teachers in Thailand’s worst school shooting in years", "language": "en", "source_name": "CNN"}, "qD3cYZVm1Uc": {"title": "AI created 16 new viruses: Why that\'s a good thing", "language": "en", "source_name": "CNN"}, "D_mF5-H2ftc": {"title": "Federal judges call out Trump over violence against judges", "language": "en", "source_name": "CNN"}, "dbJta2aw6YI": {"title": "What we know about US\' dwindling missile stockpile | Military experts on Iran", "language": "en", "source_name": "CNN"}, "vdwLrxy67Q4": {"title": "Iran says Hormuz deal is close. What would seal the deal?", "language": "en", "source_name": "CNN"}, "m-0tf51KpX4": {"title": "Foreign residents caught in Kumamoto quake aftermathーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "iyXv8SWudQI": {"title": "Hiroshima teen writing historyーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "IW9Hj-Nrwow": {"title": "Australia fights to keep fun in the water safeーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "1QIknqW1fCY": {"title": "Aung San Suu Kyi meets Red Cross officialーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "UPoyN2pluk8": {"title": "The forgotten crisis: Rohingya refugeesーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "V36uLiz7K5E": {"title": "The pickleball boom has reached JapanーNHK WORLD-JAPAN NEWS", "language": "en", "source_name": "NHK World Japan"}, "rejmFoqr_YA": {"title": "Supermarkets turn to AI to power sales, efficiency", "language": "en", "source_name": "NHK World Japan"}, "td6hm8NOZWM": {"title": "Spain and Italy\'s clash over Ceuta migrants escalates", "language": "en", "source_name": "Sky News"}, "Hb33m-qCYp8": {"title": "The repeated failures which allowed sex offender Simon Levy to kill", "language": "en", "source_name": "Sky News"}, "Op5WvxK7CCg": {"title": "El-Sayed on Trump criticism: \'I don\'t let mine go in the Oval Office\'", "language": "en", "source_name": "Sky News"}, "JhwAyo7GL9A": {"title": "Trump\'s Gaza deal: A ceasefire or a stalemate?", "language": "en", "source_name": "Sky News"}, "Hc-RnECvJJ0": {"title": "Thời sự toàn cảnh 12/8: Mỹ tuyên bố phóng tên lửa vô hiệu hóa tàu hàng hướng đến Iran | VTV24", "language": "vi", "source_name": "VTV24"}, "q4PnprL2KMg": {"title": "U.S. Deploys 20 Warships to the Middle East | News Brief | VTV24", "language": "vi", "source_name": "VTV24"}, "TAhMOIglVWU": {"title": "Russian missile attacks kill nine in Kyiv", "language": "en", "source_name": "Channel 4 News"}, "bYMyw5qSCYM": {"title": "Ceuta: Spain PM condemns EU’s ‘selfish’ response to migrant crossings", "language": "en", "source_name": "Channel 4 News"}, "gf_YGTQQbso": {"title": "Why Trump is siding with Ukraine, for now | FourSight interview", "language": "en", "source_name": "Channel 4 News"}, "2fHDYh6gYw0": {"title": "How Trump is transforming the capital of America", "language": "en", "source_name": "Channel 4 News"}, "5DHPaW5Wk_8": {"title": "Andy Burnham unveils devolution plan", "language": "en", "source_name": "Channel 4 News"}, "47EBP1i1XLc": {"title": "The tiny Spanish territory overrun by migrants", "language": "en", "source_name": "Channel 4 News"}, "f_QqArI7bAQ": {"title": "Inside the funeral home of horrors", "language": "en", "source_name": "Channel 4 News"}, "PGS1cNwOU_Y": {"title": "Stringer Dispatch: Death toll from Colombia\'s M7.5 earthquake continues to mount", "language": "en", "source_name": "CGTN"}, "xE7Ics_PgoA": {"title": "Taizhou races to restore power and clean up after Typhoon Dolphin", "language": "en", "source_name": "CGTN"}, "8FKu27jo9FI": {"title": "Thailand mourns school shooting victims amid growing scrutiny over youth gun access", "language": "en", "source_name": "CGTN"}, "Ds9O6dKcHaw": {"title": "Looking into China: A Brazilian soprano\'s resonance with Chinese culture", "language": "en", "source_name": "CGTN"}, "9acmLNvn0h0": {"title": "Think tank chief: Japan should accept China-US parity and stabilize ties", "language": "en", "source_name": "CGTN"}, "I449R6u2ebs": {"title": "From inactive to marathon runner: A 74-year-old doctor\'s prescription for lifelong fitness", "language": "en", "source_name": "CGTN"}, "qLe9Do5Xq3Q": {"title": "Rare sea turtle Rhossi returns to Texas after Wales rescue", "language": "en", "source_name": "Reuters"}, "hj0msH9OmLo": {"title": "Trump says his plane faced \'greater risk\' in flight change", "language": "en", "source_name": "Reuters"}, "eSCoVfGY12s": {"title": "Somalia battles child hunger as MSF says effort hit by funding cuts", "language": "en", "source_name": "Reuters"}, "AFv6kioBvp4": {"title": "Johannesburg\'s water crisis divides poor and rich as elections near", "language": "en", "source_name": "Reuters"}, "X_eUNsYFlM4": {"title": "Millions of Afghan girls barred from school under Taliban, UNESCO says", "language": "en", "source_name": "Reuters"}, "N1qyFghuULI": {"title": "Bondi hero Ahmed Al Ahmed’s assault charges dropped | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "3qlq5Y9sGkY": {"title": "Fmr Secret Service agent on Trump plane switch operation | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "MvDGUomKBFA": {"title": "Commonwealth Bank reports $10.9 billion annual profit | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "5BKN_JMZnGg": {"title": "Houthi ballistic missile attack in Red Sea could squeeze Saudi oil exports | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "QPxwLpkvKfs": {"title": "National police raids target illegal tobacco and vapes at over 100 petrol stations | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "ObAsFXPSAdQ": {"title": "Westpac chief economist predicts \'about a year\' before we start seeing rate cuts | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "A1tBBZ6BCog": {"title": "NZ PM Luxon faces leadership questions after series of gaffes | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "jn6OYtSBROw": {"title": "Human fridges and thermal drones: Japan and Korea\'s response to hotter summers | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "k_GSjUWJle0": {"title": "Syria\'s Bashar al-Assad sentenced to death in absentia for war crimes | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "QPRciD5RGV8": {"title": "Drought declared across two-thirds of England | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "JZrH9OwrZSM": {"title": "Trump reportedly hid in catering truck to avoid alleged Iranian threat | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "wBIzyzPCIXc": {"title": "Alan Kohler on why house prices will keep falling | ABC News Daily podcast", "language": "en", "source_name": "ABC News Australia"}, "g8O0vQcKfw8": {"title": "\'Stood in silence\': Colombia rocked by powerful 7.4 earthquake, 132 dead | The World | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "6DD7rOo9PFc": {"title": "Taiwan disrupts internet to millions, simulating invasion by China | The World | ABC News", "language": "en", "source_name": "ABC News Australia"}, "LNwWKDYZnzE": {"title": "Colombia Quake: Colombian President Tours Quake-Hit Pereira To Assess Damage | WION News", "language": "en", "source_name": "WION"}, "BEMtbRpAiY8": {"title": "Solar Eclipse 2026: Eclipse Path Stretches from Russia Through Iceland to Spain | WION Newspoint", "language": "en", "source_name": "WION"}, "eJLCShWreM8": {"title": "Libyan National Army\'s Intel Chief Assassinated on Monday; Fawzi Al-Mansouri Killed in Car Bombing", "language": "en", "source_name": "WION"}, "_YGjBX7CBA8": {"title": "Colombia Quake Death Toll Rises To 250 | Pakistan Minorities March For Rights | WION Headlines", "language": "en", "source_name": "WION"}, "Bsz-Dakq0mY": {"title": "North Korea Missile Flies 700km, Lands Off Japan | WION News", "language": "en", "source_name": "WION"}, "5EXJjHtYCck": {"title": "Colombia Quake: More Than 1600 Buildings Reported Damaged Or Collapsed | WION", "language": "en", "source_name": "WION"}, "ya-5Hy-yehs": {"title": "Afghanistan Education System Under Strain After Taliban Bars 2.4 Million Girls From School | WION", "language": "en", "source_name": "WION"}, "yZYvrj7Gbh0": {"title": "Russia-Ukraine War: Zaporizhzhia Steel Plant Halts Production After Deadly Attack | WION News", "language": "en", "source_name": "WION"}, "jUSqOlLSBXk": {"title": "Polar Bear Cub Obsessed with Traffic Cones; Japan\'s Polar Bear Goes Viral | WION", "language": "en", "source_name": "WION"}, "wqnbbwFfiRc": {"title": "Indonesian Ferry Catches Fire Near Lombok, Rescue Team Saves Survivors From Burning Ship  |Originals", "language": "en", "source_name": "WION"}, "lA5dneuzSXE": {"title": "Trump Says His Plane Faced \'Greater Risk\' | IRGC Warns of Operations Beyond Borders | WION Headlines", "language": "en", "source_name": "WION"}, "Fd2Nke0tePw": {"title": "Hormuz Deadlock Puts Oil Prices Under Pressure | World Business Watch", "language": "en", "source_name": "WION"}, "VxizzQxb9l4": {"title": "Canadians filing for insolvency at rates not seen in more than a decade | Hanomansing Tonight", "language": "en", "source_name": "CBC News"}, "TpsTf16cdzM": {"title": "Canada-U.S. negotiators hoping to pitch tariff deal to Trump next week | Power & Politics", "language": "en", "source_name": "CBC News"}, "GE2qysbIXl4": {"title": "\'We live here\': Latvia\'s defence chief on the threat of neighbour Russia | Power & Politics", "language": "en", "source_name": "CBC News"}, "SVM6yAohaUQ": {"title": "Hundreds of thousands of jobs on the line with trade talks: business council | Power & Politics", "language": "en", "source_name": "CBC News"}, "xppaKLnZPmA": {"title": "US-Iran war: 3 Pakistani citizens killed in ship attack by Houthis in Red Sea", "language": "en", "source_name": "CNA"}, "jnMTmrR3aTM": {"title": "Former Chinese premier Zhu Rongji dies aged 97", "language": "en", "source_name": "CNA"}, "5uJ0ZdS-zcM": {"title": "Yen nears 160 per US dollar: Is Japan about to intervene again?", "language": "en", "source_name": "CNA"}, "3ZK70plv3M4": {"title": "Delhi Khan Market News | ‘NDMC Is Misguided’: Khan Market Traders Oppose Anti-Encroachment Drive", "language": "en", "source_name": "NDTV"}, "EdeZO65xD_8": {"title": "Solar Eclipse 2026 | What Will Happen During Today\'s Solar Eclipse?", "language": "en", "source_name": "NDTV"}, "OkUNA5xHQCE": {"title": "Trump’s Air Force One ‘Plane Switch’ Sparks James Bond Comparison", "language": "en", "source_name": "NDTV"}, "oBwbYOYwzFQ": {"title": "West Asia War: Yemen\'s Houthis Rebel Group Kill Atleast Six and Wound 10 Others in Bab al-Mandeb", "language": "en", "source_name": "NDTV"}, "CpCNc0Dq8r0": {"title": "Colombia earthquake leaves thousands missing as rescue efforts continue", "language": "en", "source_name": "TRT World"}, "CNFaUEtWNas": {"title": "Time banking helps Hong Kong\'s ageing population", "language": "en", "source_name": "TRT World"}, "1JcafJG5NZY": {"title": "Israeli outposts rapidly expand across occupied West Bank", "language": "en", "source_name": "TRT World"}, "ZW3o3LLtR_I": {"title": "Syrian court sentences former ruler Bashar al Assad to death", "language": "en", "source_name": "TRT World"}, "t4IN6XJhBpo": {"title": "Ceuta migrant influx reignites EU immigration debate", "language": "en", "source_name": "TRT World"}, "fkF6W2CXNSM": {"title": "B.C. battles \'explosive\' wildfires as more than 20,000 remain under evacuation order", "language": "en", "source_name": "CBC News"}, "zG6UEYjHd2k": {"title": "Ryan Wedding’s alleged ‘de facto bank’ agrees to U.S. extradition", "language": "en", "source_name": "CBC News"}, "iVTjJg4DTI8": {"title": "B.C. orchard offers wildfire evacuees a \'moment of escape\'", "language": "en", "source_name": "CBC News"}, "Wwp30QTSuzI": {"title": "Large area of northwestern Ontario burned in wildfire, but crews making progress, says minister", "language": "en", "source_name": "CBC News"}, "I5LkTcg5bD8": {"title": "Trump secretly swapped planes in Turkey via catering truck because of Iran threat: reports", "language": "en", "source_name": "CBC News"}, "naBUQDrTm_M": {"title": "\'No one wants to drink a smoky wine,\' says B.C. vineyard owner in fire zone", "language": "en", "source_name": "CBC News"}, "-rDETFtnjd0": {"title": "Why Quebec is seeing a surge in unclaimed bodies", "language": "en", "source_name": "CBC News"}, "J80FvOiH30o": {"title": "Chinese tech firm Honor launches \'robophone\'", "language": "en", "source_name": "CNA"}, "xBOG9k57w_E": {"title": "How will Europe’s total solar eclipse put pressure on its energy grid?", "language": "en", "source_name": "CNA"}, "QO5xBLtAs-o": {"title": "CNA Investigates: More cases of \'hidden adults\' in Singapore emerging", "language": "en", "source_name": "CNA"}, "L_8iZSkgqoI": {"title": "AI can transform healthcare, but humans must make final decisions: WHO official", "language": "en", "source_name": "CNA"}, "3sk2j1KlrAQ": {"title": "Wildfires rage across the Okanagan | The Current", "language": "en", "source_name": "CBC News"}, "weLXbV9B7Uw": {"title": "Calgary cop shot during traffic stop, suspect found dead after manhunt | Hanomansing Tonight", "language": "en", "source_name": "CBC News"}, "wLI3Qiv0cmU": {"title": "\'Every year is going to be a bad fire year\': wildfire expert | Power & Politics", "language": "en", "source_name": "CBC News"}, "UpUwUY65uzo": {"title": "\'Beyond the rhetoric\' provinces and states need each other: Quebec minister | Power & Politics", "language": "en", "source_name": "CBC News"}, "7qySeblp0Rg": {"title": "How to avoid a dangerous bear encounter, according to an expert", "language": "en", "source_name": "CBC News"}, "iQmLdSotApI": {"title": "Why are B.C. wildfires spreading so quickly?", "language": "en", "source_name": "CBC News"}, "mFkyz1MEHx8": {"title": "Trump\'s former personal lawyer Todd Blanche sworn in as U.S. attorney general", "language": "en", "source_name": "CBC News"}, "VYnvny6WAPA": {"title": "7.4-magnitude earthquake strikes Colombia, killing at least 47", "language": "en", "source_name": "CBC News"}, "2wSYcm_MzNw": {"title": "Japan\'s matcha boom: Soaring demand creates new challenges for tea growers", "language": "en", "source_name": "CNA"}, "K-VSAqEjTHI": {"title": "Indonesia ferry fire: Authorities say 1 person dead, 211 safely evacuated amid rescue efforts", "language": "en", "source_name": "CNA"}, "mRcU6eSqRko": {"title": "What is holding back universal health coverage in Asia Pacific?", "language": "en", "source_name": "CNA"}, "e_V6XAeLdeI": {"title": "Asia’s health systems must prepare for worsening climate threats: Expert", "language": "en", "source_name": "CNA"}, "fYgYEoeoYS4": {"title": "China auto sales: Purchases fall in July for 10th straight month amid weak consumer spending", "language": "en", "source_name": "CNA"}, "aro4wTgXa-Y": {"title": "India protest: Modi\'s BJP calls for statewide shutdown over police crackdown on students", "language": "en", "source_name": "CNA"}, "dEk-32GMbkY": {"title": "Firearms controls tightened in Thailand following 2 deadly shooting incidents", "language": "en", "source_name": "CNA"}, "Wbm09_gVdqU": {"title": "Over 100 killed after 7.4-magnitude quake strikes Colombia", "language": "en", "source_name": "CNA"}, "y9BigFu0d7k": {"title": "Colombia earthquake: Damaged roads, communications complicating relief efforts, says IFRC official", "language": "en", "source_name": "CNA"}, "hfoK8ZUyQJQ": {"title": "US envoy to ASEAN reaffirms commitment to Southeast Asia amid trade concerns", "language": "en", "source_name": "CNA"}, "HqRGox3Fg4M": {"title": "Growing older, giving more: Meet two senior volunteers who have found purpose in giving back", "language": "en", "source_name": "CNA"}, "MblIMCoVG5s": {"title": "Europe, US, Canada ready for rare solar eclipse", "language": "en", "source_name": "TRT World"}, "3YaPkoTwwz0": {"title": "Rescue teams race to find survivors after Colombia quake", "language": "en", "source_name": "TRT World"}, "0bX0zXKOKhw": {"title": "Ukraine steps up long-range drone strikes inside Russia", "language": "en", "source_name": "TRT World"}, "TfR7i2qZnys": {"title": "Iran’s historical sites bear the scars of Israeli strikes", "language": "en", "source_name": "TRT World"}, "vE036F_imO0": {"title": "At least 132 people killed following powerful quake in Colombia", "language": "en", "source_name": "TRT World"}, "T_Q4xbECFnc": {"title": "Illegal Israeli settler violence, home demolitions at record highs", "language": "en", "source_name": "TRT World"}, "IlbxRU4BulI": {"title": "Clashes erupt during student protests in India’s Jharkhand state", "language": "en", "source_name": "TRT World"}, "-PoN_dB0ApE": {"title": "EU scientists say western Europe saw warmest June-July on record", "language": "en", "source_name": "TRT World"}, "OQWjk-VZrxw": {"title": "Displaced Palestinians face gunfire near Israeli buffer zone", "language": "en", "source_name": "TRT World"}, "vrbPFo0joRc": {"title": "Drones curb Russian advance on Ukrainian front line", "language": "en", "source_name": "TRT World"}, "ie6u23M4Qzc": {"title": "NZ PM Luxon survives confidence vote but leadership questions remain | The World | ABC News", "language": "en", "source_name": "ABC News Australia"}, "dpAO_LBst8c": {"title": "Ebola death toll in DR Congo exceeds 2,000 amid fears virus is mutating | The World | ABC News", "language": "en", "source_name": "ABC News Australia"}, "xvgMw17BY_w": {"title": "Singapore\'s elderly take up parkour to stay healthy and survive falls | The World | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "3puJFW89RIY": {"title": "Japanese govt downplays \'melons\' gaffe, opposition unveils new housing policy | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "8jrid0Q-CO0": {"title": "Queensland man charged with domestic violence murder of former partner | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "X2xARZC43xI": {"title": "Colombia quake death toll soars above 250 with thousands missing | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "vO2-VAY_JwM": {"title": "Commonwealth Bank shares fall in wake of annual results | Finance Report | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "U6xH1yMxKyg": {"title": "Acclaimed photographer sentenced for ‘serious’, ‘insidious’ DV offending | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "iHeRH47w9gs": {"title": "Federal Labor MP joins calls for youth curfew in Katherine | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "_jAYaJDQBr0": {"title": "Rescuers search through rubble for survivors of major quake in Colombia | ABC NEWS", "language": "en", "source_name": "ABC News Australia"}, "Uc9H3XdvbHg": {"title": "Mount Etna Erupts In Spectacular Display As Glowing Lava Lights Up Sicily | WION", "language": "en", "source_name": "WION"}, "oOhSdtYtJ1Q": {"title": "Kazakhstan Builds New Highway to Reshape China-Europe Trade, Cut Transit Route by 1,000 Km", "language": "en", "source_name": "WION"}, "U7Dra8U1d-Y": {"title": "Asian Markets Face $25.5 Billion Foreign Outflow as Taiwan, South Korea Lead Tech Sell-Off", "language": "en", "source_name": "WION"}, "GoQm-v4aS7k": {"title": "Russia Unleashes Unstoppable Oniks Missile to Sink Warships | WION Originals", "language": "en", "source_name": "WION"}, "_4wEs8cLmu8": {"title": "Colombia Earthquake: Rescue Teams Race to Find Survivors as Death Toll Nears 250", "language": "en", "source_name": "WION"}, "SnwZ1kCQT2A": {"title": "Iran Warns of Long War as IRGC Shifts to Offensive Doctrine | Trump Faces Tougher Choices | US |Iran", "language": "en", "source_name": "WION"}, "jeNs5lo-1M8": {"title": "Pharmaceutical company Eli Lilly wants deal to bring Mounjaro to Australia under PBS | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "vJnmRqMkMg4": {"title": "Liberal Senator wishes he’d done more to ‘stop the madness’ in Morrison government | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "CjEUqlEMJi8": {"title": "Albanese government under pressure to quickly fix widow tax | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "7eR9Zdxgqto": {"title": "Sussan Ley quit politics and found herself on a Greyhound bus travelling across America | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "BJZb8VcjiLc": {"title": "Antisemitism royal commission turns its attention to protest movements and the far-right | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "umaWlRJg9bE": {"title": "Why Albanese\'s podcast comments have made headlines again | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "DQzsT163lew": {"title": "How Researchers Tried to Combat SARS (full documentary) | FRONTLINE/World", "language": "en", "source_name": "FRONTLINE PBS"}, "FoI__LOQ36M": {"title": "How a Mexican Migrant Farmer’s Journey to the U.S. Turned Fatal (full documentary) | FRONTLINE/World", "language": "en", "source_name": "FRONTLINE PBS"}, "6Iev0NaTTSo": {"title": "How Rats Helped Save Human Lives | Tanzania: Hero Rats (full documentary) | FRONTLINE/World", "language": "en", "source_name": "FRONTLINE PBS"}, "e0fwgjRdQH8": {"title": "When Cable TV Came to a Buddhist Kingdom (full documentary) | FRONTLINE/World", "language": "en", "source_name": "FRONTLINE PBS"}, "kIRP3hZJuG8": {"title": "Inside a Changing Nation | Vietnam: Looking for Home (full documentary) | FRONTLINE/World", "language": "en", "source_name": "FRONTLINE PBS"}, "dfp2TAoG92Q": {"title": "How Agencies & Brokers Lure Pregnant Women for ‘Adoption Tourism’ | FRONTLINE + @RETROREPORT", "language": "en", "source_name": "FRONTLINE PBS"}, "TugPBKBXIHc": {"title": "Why China and America see AI differently | The Economist", "language": "en", "source_name": "The Economist"}, "A5yH4PJX__o": {"title": "Who’s really running Iran? | The Economist", "language": "en", "source_name": "The Economist"}, "MzRgt8Kv7SE": {"title": "Is billionaire wealth becoming more legitimate? | The Economist", "language": "en", "source_name": "The Economist"}, "thGyYY1wqYk": {"title": "What can Big Macs tell you about the global economy? | The Economist", "language": "en", "source_name": "The Economist"}, "emUjVtgBGBg": {"title": "Do billionaires deserve their wealth? | The Economist", "language": "en", "source_name": "The Economist"}, "j16oj0DPv7Q": {"title": "Why knowing the future doesn\'t always help predict markets | The Economist", "language": "en", "source_name": "The Economist"}, "gNk5UC3hAHc": {"title": "Why the AI race won\'t have a winner | The Economist", "language": "en", "source_name": "The Economist"}, "xjhywQlzJYo": {"title": "How AI Is Transforming Modern Warfare | Proton Data Wars | VICE News", "language": "en", "source_name": "VICE News"}, "6DjrqMrCeC0": {"title": "\\"Our Weapon Is AI\\" | Iran\'s Viral Meme Warriors Interviewed by Shane Smith & Andrew Callaghan", "language": "en", "source_name": "VICE News"}, "jh_3ni93yIo": {"title": "Survivors uncover new evidence in Australia’s contaminated blood scandal | Australian Story", "language": "en", "source_name": "ABC News In-depth"}, "T0cmWnGID2k": {"title": "Could the South China Sea become the next global battleground? | Foreign Correspondent", "language": "en", "source_name": "ABC News In-depth"}, "PcnSL3I2o6Q": {"title": "Did staff shortages cause this near-miss at Australia\'s busiest airport? | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "DTkg_6XcG24": {"title": "The 2026 census is mandatory – here\'s what you need to know | ABC NEWS Loop", "language": "en", "source_name": "ABC News In-depth"}, "ByDyG7W45YM": {"title": "Kyrgyzstan - The erosion of democracy | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "WpeJItcFW7c": {"title": "BookTok and Gen Z - Are young people saving the book market? | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "rkMBZ2bCY6Y": {"title": "JD Vance: From Trump critic to vice president | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "Hb9YEfOV9sc": {"title": "Hotel hope - Where refugees begin a new life | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "4XjVsXnqS_o": {"title": "Assaults in the hospital - When doctors harass their female colleagues | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "CRZ3Ml4ww8k": {"title": "Caught in India’s AIDS Crisis | India: The Sex Workers (full documentary) | FRONTLINE/World", "language": "en", "source_name": "FRONTLINE PBS"}, "vf8O33qLh4c": {"title": "Fmr. Trump envoy on negotiating the Abraham Accords | Jason Greenblatt (interview) | FRONTLINE", "language": "en", "source_name": "FRONTLINE PBS"}, "g-T1boiZFfM": {"title": "The final filmed interview with Sir Alex Younger, the late MI6 chief | The Economist", "language": "en", "source_name": "The Economist"}, "Ndp_JcjwFN4": {"title": "Why Le Pen is keeping Trump at a distance | The Economist", "language": "en", "source_name": "The Economist"}, "jpnTQzPHewQ": {"title": "A full-length interview with Tucker Carlson | The Economist", "language": "en", "source_name": "The Economist"}, "JuIsRJDzgwQ": {"title": "What happens if Binyamin Netanyahu loses Israel’s next election? | The Economist", "language": "en", "source_name": "The Economist"}, "G3CSuUeuyJg": {"title": "1897: To the North Pole in a hydrogen balloon | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "cJPpfoC0D1I": {"title": "Steve Jobs and the Apple story - From rebel to tech titan | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "b5C6x7V6TMU": {"title": "Can future pandemics be prevented? | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "R1avfIHO668": {"title": "From ADHD to heart disease - Why women get sick differently than men | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "cP6ufEYIig8": {"title": "Beirut explosion 2020: The unsolved catastrophe | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "7ZhpAPq1Ikg": {"title": "Endangered Amazon - Why the Earth\'s lungs are dying | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "T9-Yt4IZWS4": {"title": "Neele dances through life - Artist with Down syndrome follows her own path | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "XtBuPjp-Zm4": {"title": "When shame changes sides - The case of Gisèle Pelicot | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "BA1yL8RfbME": {"title": "Morocco\'s cannabis reform: Why small farmers are left behind | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "SLntIypiacE": {"title": "Will non-alcoholic wine save the winemakers? | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "uWSuGqi_Lo8": {"title": "By train to Mount Etna | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "ogH8-nNxUX0": {"title": "The Milwaukee Cannibal - \\"My Friend Dahmer\\" | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "jDP2hcOCGcM": {"title": "The Berlin rental rip-off: The housing crisis boom on tiny rooms  | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "CLfVOyzTJOU": {"title": "For-Profit Adoptions & Misconduct Allegations (full documentary) | FRONTLINE (PBS) + Retro Report", "language": "en", "source_name": "FRONTLINE PBS"}, "HbwIIQQ-yBE": {"title": "President Trump’s Pressure Campaign on the Federal Reserve (full documentary) | FRONTLINE (PBS)", "language": "en", "source_name": "FRONTLINE PBS"}, "Uj_QwdSEBks": {"title": "Police Bombed MOVE Organizers’ Home in 1985 (full documentary) | FRONTLINE (PBS)", "language": "en", "source_name": "FRONTLINE PBS"}, "ZgXifr5iLbU": {"title": "Immigration Under President Trump’s First Term | Kids Caught in the Crackdown (full documentary)", "language": "en", "source_name": "FRONTLINE PBS"}, "MbPBVFRJQmk": {"title": "Protesters, Bystanders Arrested | Caught in the Crackdown (documentary) | FRONTLINE + ProPublica", "language": "en", "source_name": "FRONTLINE PBS"}, "FuOnV9FGH5U": {"title": "The Deal: Trump, Bukele & the Gangs of El Salvador (full documentary) | FRONTLINE (PBS) + El Faro", "language": "en", "source_name": "FRONTLINE PBS"}, "2pXd4VnXllc": {"title": "Investigating Bukele and Trump’s Controversial CECOT Deal | FRONTLINE (PBS) + @elfaro", "language": "en", "source_name": "FRONTLINE PBS"}, "td5Iph8BFgE": {"title": "Strike on Iran: The Nuclear Question (updated documentary) | FRONTLINE (PBS)", "language": "en", "source_name": "FRONTLINE PBS"}, "g1RUzj3QwUg": {"title": "Tracking Suspected Members of the Pakistani Taliban | Taliban Hunters (full documentary) | FRONTLINE", "language": "en", "source_name": "FRONTLINE PBS"}, "Rz1mVvf6M8I": {"title": "What Iran’s top security official said about June 2025 U.S. and Israeli strikes", "language": "en", "source_name": "FRONTLINE PBS"}, "t0uOd5i2a5E": {"title": "Inside a Maximum Security Prison’s Isolation Unit | Solitary Nation (full documentary) | FRONTLINE", "language": "en", "source_name": "FRONTLINE PBS"}, "M6rKtOVridE": {"title": "What makes us happier? - The formula for happiness | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "QXFkB_o5vX8": {"title": "Nigeria’s mysterious art treasures | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "z0bEypacaAA": {"title": "Muscle mania and the cult of masculinity - When the body becomes an obsession | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "TRCj38jjAsM": {"title": "Forensics 3.0 - How modern crime labs are closing in on criminals | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "kXUOrxsR5VI": {"title": "How does sexual desire really arise? - The science behind arousal | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "gSV7RlgQtWc": {"title": "Flatpack forests - IKEA’s practices in the Carpathian Mountains | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "KsBib2BrM-A": {"title": "Illegal dolphin hunting | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "wYkfiTenjhU": {"title": "How the Mafia’s cattle are terrorizing Italian villages | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "BfyV2LmcyjM": {"title": "From Latin America to Europe - The dangerous schemes of human traffickers | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "1aB_6zJ72rk": {"title": "How Assad Moved to Crush Syria’s Uprising | The Regime (full documentary) | FRONTLINE (PBS)", "language": "en", "source_name": "FRONTLINE PBS"}, "BThiONABge4": {"title": "Is Australia falling behind in the AI race? | Insiders On Background", "language": "en", "source_name": "ABC News In-depth"}, "T-ocxdAzXp0": {"title": "Meet Nigel Farage’s Golden Boy: “Posh George” | ABC NEWS In-depth", "language": "en", "source_name": "ABC News In-depth"}, "t-fmcVJbGDw": {"title": "Meet the projectionist bringing The Odyssey to the big screen at Melbourne\'s IMAX | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "HPVkwkkwsro": {"title": "The secret recordings played in the NSW Liberal Party ICAC inquiry | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "_aujksUSi1c": {"title": "Alan Jones\'s indecent assault trial is underway in Sydney | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "DadOMLz_OpM": {"title": "Victoria\'s corruption watchdog delivers its verdict on the Andrews government | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "Jh7dOhbrCUQ": {"title": "Why Netanyahu is running out of options on Gaza | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "kwaQvoOtJmQ": {"title": "Has the AI boom entered a manic new phase? | The Economist", "language": "en", "source_name": "The Economist"}, "4OL6Gds1stU": {"title": "Why European oil companies are doing better than America\'s | The Economist", "language": "en", "source_name": "The Economist"}, "f53Fqt2UhVE": {"title": "Why America is stepping away from the institutions it built | The Economist", "language": "en", "source_name": "The Economist"}, "MQ6Ab5UkvQw": {"title": "Should Latin America replicate El Salvador’s President Bukele’s crackdown on gangs? | The Economist", "language": "en", "source_name": "The Economist"}, "VdLvWLroFOA": {"title": "How SpaceX\'s $3trn valuation affects the whole market  | The Economist", "language": "en", "source_name": "The Economist"}, "sfPb0vFUQow": {"title": "What does the Iran deal mean for Israel? | The Economist", "language": "en", "source_name": "The Economist"}, "-eYblJ8P6r4": {"title": "Can Donald Trump secure a deal with Cuba? | The Economist", "language": "en", "source_name": "The Economist"}, "1xkXCem4D7o": {"title": "From bees to robots: The new power of swarm intelligence | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "g-Zmy63uN9M": {"title": "Banned weapons - Why Europe is bringing back landmines | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "dOZR9OY0Qjk": {"title": "Canada\'s multicultural metropolis: When night falls in Toronto | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "OikpKh-4osA": {"title": "In the shadow of the World Cup - Mexican families search for their loved ones | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "RPezCeAlhrs": {"title": "Inside Jesse Jackson’s 1988 Presidential Campaign| Running with Jesse (full documentary) | FRONTLINE", "language": "en", "source_name": "FRONTLINE PBS"}, "_OuloHpTNyo": {"title": "Parents of children with autism concerned about future of Queensland services | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "tjrc-qbYecE": {"title": "Gambling companies deny offering drugs and escorts to customers | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "81eUG-x34lk": {"title": "Inside Albania\'s Flamingo Revolution | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "vCYxZz_sFY0": {"title": "Tokyo\'s Geisha: Inside the closed world of Japan’s Enigmatic Icons | Foreign Correspondent", "language": "en", "source_name": "ABC News In-depth"}, "0qVcT-J1nt0": {"title": "The peptide black market hiding in plain sight on social media | Media Watch", "language": "en", "source_name": "ABC News In-depth"}, "1kQSDVe8deM": {"title": "Courts begin to deliver justice after Islamic State genocide against Yazidis | 7.30", "language": "en", "source_name": "ABC News In-depth"}, "P1W9Pakd_6o": {"title": "Why are young Indians joining the Cockroach Janta Party? | The Economist", "language": "en", "source_name": "The Economist"}, "pn9x8V4SHzM": {"title": "Can the US-Iran ceasefire hold? | The Economist", "language": "en", "source_name": "The Economist"}, "F9AykvYmrCo": {"title": "Why are Gen-Z socialists obsessed with taxing billionaires? | The Economist", "language": "en", "source_name": "The Economist"}, "Bd9hGy-cgsE": {"title": "Are there good and bad billionaires? | The Economist", "language": "en", "source_name": "The Economist"}, "k4Dk9xl7sgU": {"title": "Is Colombia heading for its most polarised election yet? | The Economist", "language": "en", "source_name": "The Economist"}, "vjx8TyadOKY": {"title": "Why are AI data centres facing a backlash? | The Economist", "language": "en", "source_name": "The Economist"}, "RAwiu_efcMk": {"title": "Wild Rügen - Tourism vs. nature conservation | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "RsaItlC-iq8": {"title": "San Michele - Venice\'s cemetery island | DW Documentary", "language": "en", "source_name": "DW Documentary"}, "HXIXzopaQpg": {"title": "Displaced on Java - A village battling rising sea levels | DW Documentary", "language": "en", "source_name": "DW Documentary"}}')
print(f'Loaded {len(VIDEO_METADATA)} video metadata entries.')


In [ ]:
# Locate newsseek_manifest.json
search_roots = [Path('/kaggle/input'), Path('../datasets'), Path('./datasets'), Path('./')]
manifest_path = None
for r in search_roots:
    matches = list(r.glob('**/newsseek_manifest.json'))
    if matches:
        manifest_path = matches[0]
        break

if not manifest_path:
    raise FileNotFoundError('newsseek_manifest.json not found in search paths!')

print(f'Manifest located at: {manifest_path}')
with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest_raw = json.load(f)

video_records = list(manifest_raw.values()) if isinstance(manifest_raw, dict) else manifest_raw
print(f'Total videos in manifest: {len(video_records)}')

# Find keyframes directory
base_dataset_dir = manifest_path.parent
print(f'Base dataset dir: {base_dataset_dir}')

# Flatten all keyframes into a structured list
all_keyframes = []
for v in video_records:
    vid = v['video_id']
    v_info = VIDEO_METADATA.get(vid, {'title': v.get('video_file', vid), 'language': 'en', 'source_name': ''})
    for kf in v.get('keyframes', []):
        kf_rel = kf['file_path']
        cand1 = base_dataset_dir / kf_rel
        cand2 = base_dataset_dir / 'newsseek_keyframes' / kf_rel
        cand3 = base_dataset_dir / Path(kf_rel).name
        
        actual_path = None
        if cand1.exists():
            actual_path = cand1
        elif cand2.exists():
            actual_path = cand2
        elif cand3.exists():
            actual_path = cand3
        else:
            actual_path = cand1 # fallback
            
        all_keyframes.append({
            'kf_idx': len(all_keyframes),
            'video_id': vid,
            'frame_number': kf['frame_number'],
            'timestamp_sec': kf['timestamp_sec'],
            'scene_index': kf['scene_index'],
            'file_path': str(actual_path),
            'relative_path': kf_rel,
            'video_title': v_info['title'],
            'language': v_info['language'],
        })

print(f'Flattened {len(all_keyframes)} total keyframes across {len(video_records)} videos.')
sample_kf = all_keyframes[0]
print(f'Sample Keyframe: {sample_kf["relative_path"]}, exists: {Path(sample_kf["file_path"]).exists()}')


In [ ]:
def extract_tensor_features(output):
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, 'pooler_output') and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, 'last_hidden_state') and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]
    if hasattr(output, '__getitem__'):
        return output[0]
    return output

class KeyframeDataset(Dataset):
    def __init__(self, keyframe_list, processor, is_siglip=False):
        self.items = keyframe_list
        self.processor = processor
        self.is_siglip = is_siglip
        self.fallback_img = Image.new('RGB', (224, 224), color=(128, 128, 128))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        p = self.items[idx]['file_path']
        try:
            img = Image.open(p).convert('RGB')
        except Exception:
            img = self.fallback_img
        return img

def collate_fn_clip(batch):
    return clip_processor(images=batch, return_tensors='pt')

def collate_fn_siglip(batch):
    return siglip_processor(images=batch, return_tensors='pt')

print('Defined KeyframeDataset and Collate functions.')


In [ ]:
# Load CLIP ViT-B/32
print('--> Loading openai/clip-vit-base-patch32...')
clip_model_id = 'openai/clip-vit-base-patch32'
clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
clip_model = CLIPModel.from_pretrained(clip_model_id).to(device).eval()

clip_loader = DataLoader(
    KeyframeDataset(all_keyframes, clip_processor),
    batch_size=256,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn_clip,
    pin_memory=True,
)

clip_kf_embeds = []
t0 = time.time()
with torch.no_grad():
    for batch in tqdm(clip_loader, desc='CLIP Keyframes'):
        inputs = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feats = extract_tensor_features(clip_model.get_image_features(**inputs))
            feats = feats / feats.norm(dim=-1, keepdim=True)
        clip_kf_embeds.append(feats.cpu().float())

clip_kf_tensor = torch.cat(clip_kf_embeds, dim=0)
clip_kf_time = time.time() - t0
print(f'CLIP Keyframes: {len(clip_kf_tensor)} in {clip_kf_time:.2f}s ({len(clip_kf_tensor)/clip_kf_time:.1f} fps)')

# Encode Video Titles (Queries)
unique_vids = [v['video_id'] for v in video_records]
unique_titles = [VIDEO_METADATA.get(v, {}).get('title', v) for v in unique_vids]
print(f'Encoding {len(unique_titles)} video queries with CLIP...')

clip_txt_embeds = []
with torch.no_grad():
    for i in range(0, len(unique_titles), 128):
        batch_txt = unique_titles[i : i + 128]
        inputs = clip_processor(text=batch_txt, padding=True, truncation=True, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feats = extract_tensor_features(clip_model.get_text_features(**inputs))
            feats = feats / feats.norm(dim=-1, keepdim=True)
        clip_txt_embeds.append(feats.cpu().float())

clip_txt_tensor = torch.cat(clip_txt_embeds, dim=0)
del clip_model, clip_processor
torch.cuda.empty_cache()


In [ ]:
# Load SigLIP ViT-B/16-224
print('--> Loading google/siglip-base-patch16-224...')
siglip_model_id = 'google/siglip-base-patch16-224'
siglip_processor = AutoProcessor.from_pretrained(siglip_model_id)
siglip_model = AutoModel.from_pretrained(siglip_model_id).to(device).eval()

siglip_loader = DataLoader(
    KeyframeDataset(all_keyframes, siglip_processor, is_siglip=True),
    batch_size=256,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn_siglip,
    pin_memory=True,
)

siglip_kf_embeds = []
t0 = time.time()
with torch.no_grad():
    for batch in tqdm(siglip_loader, desc='SigLIP Keyframes'):
        inputs = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feats = extract_tensor_features(siglip_model.get_image_features(**inputs))
            feats = feats / feats.norm(dim=-1, keepdim=True)
        siglip_kf_embeds.append(feats.cpu().float())

siglip_kf_tensor = torch.cat(siglip_kf_embeds, dim=0)
siglip_kf_time = time.time() - t0
print(f'SigLIP Keyframes: {len(siglip_kf_tensor)} in {siglip_kf_time:.2f}s ({len(siglip_kf_tensor)/siglip_kf_time:.1f} fps)')

# Encode Video Titles
siglip_txt_embeds = []
with torch.no_grad():
    for i in range(0, len(unique_titles), 128):
        batch_txt = unique_titles[i : i + 128]
        inputs = siglip_processor(text=batch_txt, padding='max_length', truncation=True, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feats = extract_tensor_features(siglip_model.get_text_features(**inputs))
            feats = feats / feats.norm(dim=-1, keepdim=True)
        siglip_txt_embeds.append(feats.cpu().float())

siglip_txt_tensor = torch.cat(siglip_txt_embeds, dim=0)
del siglip_model, siglip_processor
torch.cuda.empty_cache()


In [ ]:
# Build FAISS vector stores for NewsSeek
out_dir = Path('/kaggle/working')
storage_dir = out_dir / 'storage' / 'newsseek'
clip_store_dir = storage_dir / 'clip'
siglip_store_dir = storage_dir / 'siglip'
clip_store_dir.mkdir(parents=True, exist_ok=True)
siglip_store_dir.mkdir(parents=True, exist_ok=True)

# 1. Save raw numpy arrays
np.save(out_dir / 'newsseek_clip_keyframes.npy', clip_kf_tensor.numpy())
np.save(out_dir / 'newsseek_clip_titles.npy', clip_txt_tensor.numpy())
np.save(out_dir / 'newsseek_siglip_keyframes.npy', siglip_kf_tensor.numpy())
np.save(out_dir / 'newsseek_siglip_titles.npy', siglip_txt_tensor.numpy())
print('Saved .npy embedding arrays.')

# 2. Build metadata list
metadata_list = []
for kf in all_keyframes:
    metadata_list.append({
        'id': f'newsseek_{kf["kf_idx"]}',
        'kf_idx': kf['kf_idx'],
        'video_id': kf['video_id'],
        'frame_number': kf['frame_number'],
        'timestamp_sec': kf['timestamp_sec'],
        'scene_index': kf['scene_index'],
        'relative_path': kf['relative_path'],
        'video_title': kf['video_title'],
        'language': kf['language'],
    })

# 3. Build & Save CLIP FAISS Store (166k vectors)
clip_index = faiss.IndexFlatIP(512)
clip_vecs = np.ascontiguousarray(clip_kf_tensor.numpy(), dtype=np.float32)
faiss.normalize_L2(clip_vecs)
clip_index.add(clip_vecs)
faiss.write_index(clip_index, str(clip_store_dir / 'index.faiss'))

clip_meta_payload = {
    'dim': 512,
    'index_type': 'FlatIP',
    'model_name': 'clip',
    'count': len(metadata_list),
    'items': metadata_list,
}
with open(clip_store_dir / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(clip_meta_payload, f, ensure_ascii=False, indent=2)
print(f'Saved CLIP VectorStore ({clip_index.ntotal} vectors) -> {clip_store_dir}')

# 4. Build & Save SigLIP FAISS Store (166k vectors)
siglip_index = faiss.IndexFlatIP(768)
siglip_vecs = np.ascontiguousarray(siglip_kf_tensor.numpy(), dtype=np.float32)
faiss.normalize_L2(siglip_vecs)
siglip_index.add(siglip_vecs)
faiss.write_index(siglip_index, str(siglip_store_dir / 'index.faiss'))

siglip_meta_payload = {
    'dim': 768,
    'index_type': 'FlatIP',
    'model_name': 'siglip',
    'count': len(metadata_list),
    'items': metadata_list,
}
with open(siglip_store_dir / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(siglip_meta_payload, f, ensure_ascii=False, indent=2)
print(f'Saved SigLIP VectorStore ({siglip_index.ntotal} vectors) -> {siglip_store_dir}')

# 5. Pre-package ZIP archive
zip_path = out_dir / 'newsseek_vector_stores.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(clip_store_dir / 'index.faiss', 'storage/newsseek/clip/index.faiss')
    zf.write(clip_store_dir / 'metadata.json', 'storage/newsseek/clip/metadata.json')
    zf.write(siglip_store_dir / 'index.faiss', 'storage/newsseek/siglip/index.faiss')
    zf.write(siglip_store_dir / 'metadata.json', 'storage/newsseek/siglip/metadata.json')
print(f'Pre-packaged initial vector stores archive ({zip_path.stat().st_size / (1024**2):.2f} MB) -> {zip_path}')


In [ ]:
def evaluate_newsseek_retrieval(kf_tensor, txt_tensor, video_records, all_keyframes, k_vals=[1, 5, 10]):
    print('Evaluating Video-Wise Max-Pooling Retrieval...')
    # Step 1: Compute similarity between all keyframes and all video queries
    frame_sims = torch.matmul(kf_tensor, txt_tensor.T)
    
    N_vid = len(video_records)
    video_sim_matrix = torch.zeros((N_vid, N_vid))
    
    # Step 2: Max-Pool keyframes for each video
    kf_start = 0
    for v_idx, v in enumerate(video_records):
        n_kf = len(v.get('keyframes', []))
        if n_kf > 0:
            v_kf_sims = frame_sims[kf_start : kf_start + n_kf] # shape: (n_kf, N_vid)
            video_sim_matrix[v_idx] = v_kf_sims.max(dim=0).values
        kf_start += n_kf
        
    # Step 3: Compute Recall@K and MRR
    top_max = max(k_vals)
    _, topk_preds = video_sim_matrix.topk(top_max, dim=0, largest=True)
    targets = torch.arange(N_vid).unsqueeze(0).expand_as(topk_preds)
    matches = (topk_preds == targets)
    
    res = {}
    for k in k_vals:
        res[f'R@{k}'] = round(float((matches[:k].any(dim=0).float().mean().item()) * 100.0), 2)
        
    _, full_sorted = video_sim_matrix.sort(dim=0, descending=True)
    targets_full = torch.arange(N_vid).unsqueeze(0).expand_as(full_sorted)
    match_coords = (full_sorted == targets_full).nonzero(as_tuple=False)
    ranks = match_coords[:, 0] + 1
    res['MRR'] = round(float((1.0 / ranks.float()).mean().item()), 4)
    res['mR'] = round(float(np.mean([res[f'R@{k}'] for k in k_vals])), 2)
    
    # Modality Gap
    gap = torch.norm(kf_tensor.mean(dim=0) - txt_tensor.mean(dim=0), p=2).item()
    res['modality_gap'] = round(float(gap), 4)
    
    # Multilingual breakdown
    langs = [VIDEO_METADATA.get(v['video_id'], {}).get('language', 'en') for v in video_records]
    en_indices = [i for i, l in enumerate(langs) if l == 'en']
    non_en_indices = [i for i, l in enumerate(langs) if l != 'en']
    
    if en_indices:
        res['R@1_EN'] = round(float((matches[0, en_indices].float().mean().item()) * 100.0), 2)
    if non_en_indices:
        res['R@1_NonEN'] = round(float((matches[0, non_en_indices].float().mean().item()) * 100.0), 2)
        
    return res

clip_newsseek_bm = {}
try:
    print('\n=== EVALUATING CLIP ON NEWSSEEK ===')
    clip_newsseek_bm = evaluate_newsseek_retrieval(clip_kf_tensor, clip_txt_tensor, video_records, all_keyframes)
    clip_newsseek_bm['kf_enc_time_sec'] = round(clip_kf_time, 2)
    print(json.dumps(clip_newsseek_bm, indent=2))
except Exception as e:
    print(f'Warning: CLIP evaluation failed: {e}')
    clip_newsseek_bm = {'error': str(e), 'kf_enc_time_sec': round(clip_kf_time, 2)}

siglip_newsseek_bm = {}
try:
    print('\n=== EVALUATING SIGLIP ON NEWSSEEK ===')
    siglip_newsseek_bm = evaluate_newsseek_retrieval(siglip_kf_tensor, siglip_txt_tensor, video_records, all_keyframes)
    siglip_newsseek_bm['kf_enc_time_sec'] = round(siglip_kf_time, 2)
    print(json.dumps(siglip_newsseek_bm, indent=2))
except Exception as e:
    print(f'Warning: SigLIP evaluation failed: {e}')
    siglip_newsseek_bm = {'error': str(e), 'kf_enc_time_sec': round(siglip_kf_time, 2)}

# Save benchmark summary
newsseek_bm = {
    'dataset': 'newsseek-processed-keyframes',
    'total_videos': len(video_records),
    'total_keyframes': len(all_keyframes),
    'clip_vit_b32': clip_newsseek_bm,
    'siglip_vit_b16_224': siglip_newsseek_bm,
}
with open(out_dir / 'newsseek_benchmark_results.json', 'w', encoding='utf-8') as f:
    json.dump(newsseek_bm, f, indent=2)

# Append benchmark json to zip archive
with zipfile.ZipFile(zip_path, 'a') as zf:
    zf.write(out_dir / 'newsseek_benchmark_results.json', 'newsseek_benchmark_results.json')

print(f'Final download archive updated ({zip_path.stat().st_size / (1024**2):.2f} MB) -> {zip_path}')
print('=== NEWSSEEK INGESTION & BENCHMARK COMPLETE ===')
